# Import Packages

In [1]:
import pandas as pd
import numpy as np
import matplotlib as mpl 
import pybaseball as pb
from pybaseball import statcast
import sklearn
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_squared_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from datetime import date
import requests
from bs4 import BeautifulSoup
from pandasql import sqldf
import duckdb


# Load and Clean Data

In [ ]:
path18 = pd.read_csv('/Users/owendrummond/Downloads/gl2010_19/gl2018.txt', header=None)
path19 = pd.read_csv('/Users/owendrummond/Downloads/gl2010_19/gl2019.txt', header=None)
path20 = pd.read_csv('/Users/owendrummond/Downloads/gl2020_23/gl2020.txt', header=None)
path21 = pd.read_csv('/Users/owendrummond/Downloads/gl2020_23/gl2021.txt', header=None)
path22 = pd.read_csv('/Users/owendrummond/Downloads/gl2020_23/gl2022.txt', header=None)
path23 = pd.read_csv('/Users/owendrummond/Downloads/gl2020_23/gl2023.txt', header=None)
path24 = pd.read_csv('/Users/owendrummond/Downloads/gl2024.txt', header=None)
path25 = pd.read_csv('/Users/owendrummond/Downloads/gl2025.txt', header=None)

In [113]:
# This function adds names to columns from the raw data download, and adds a HomeWin and AwayWin indicator

def clean_data_box(df):
    rename_map = {
        0: 'Date',
        2: 'DayOfWeek',
        3: 'AwayTeam',
        4: 'AwayLeague',
        5: 'AwayGameNum',
        6: 'HomeTeam',
        7: 'HomeLeague',
        8: 'HomeGameNum',
        9: 'AwayScore',
        10: 'HomeScore',
        11: 'OutTotal',
        12: 'Day/Night',
        16: 'ParkID',
        17: 'Attendence',
        18: 'LengthMin',
        19: 'AwayLineScore',
        20: 'HomeLineScore',
        38: 'AwayPitchersUsed',
        66: 'HomePitchersUsed',
        101: 'AwayPitcherID',
        102: 'AwayPitcher',
        103: 'HomePitcherID',
        104: 'HomePitcher',

        # Away batters
        105: 'AwayBatter1ID', 106: 'AwayBatter1', 107: 'AwayBatter1Pos',
        108: 'AwayBatter2ID', 109: 'AwayBatter2', 110: 'AwayBatter2Pos',
        111: 'AwayBatter3ID', 112: 'AwayBatter3', 113: 'AwayBatter3Pos',
        114: 'AwayBatter4ID', 115: 'AwayBatter4', 116: 'AwayBatter4Pos',
        117: 'AwayBatter5ID', 118: 'AwayBatter5', 119: 'AwayBatter5Pos',
        120: 'AwayBatter6ID', 121: 'AwayBatter6', 122: 'AwayBatter6Pos',
        123: 'AwayBatter7ID', 124: 'AwayBatter7', 125: 'AwayBatter7Pos',
        126: 'AwayBatter8ID', 127: 'AwayBatter8', 128: 'AwayBatter8Pos',
        129: 'AwayBatter9ID', 130: 'AwayBatter9', 131: 'AwayBatter9Pos',

        # Home batters
        132: 'HomeBatter1ID', 133: 'HomeBatter1', 134: 'HomeBatter1Pos',
        135: 'HomeBatter2ID', 136: 'HomeBatter2', 137: 'HomeBatter2Pos',
        138: 'HomeBatter3ID', 139: 'HomeBatter3', 140: 'HomeBatter3Pos',
        141: 'HomeBatter4ID', 142: 'HomeBatter4', 143: 'HomeBatter4Pos',
        144: 'HomeBatter5ID', 145: 'HomeBatter5', 146: 'HomeBatter5Pos',
        147: 'HomeBatter6ID', 148: 'HomeBatter6', 149: 'HomeBatter6Pos',
        150: 'HomeBatter7ID', 151: 'HomeBatter7', 152: 'HomeBatter7Pos',
        153: 'HomeBatter8ID', 154: 'HomeBatter8', 155: 'HomeBatter8Pos',
        156: 'HomeBatter9ID', 157: 'HomeBatter9', 158: 'HomeBatter9Pos',

        160: 'Finished'
    }

    df = df.rename(columns=rename_map)

    drop_cols = (
        [1, 13, 14, 15, 159] +
        list(range(21, 38)) +
        [39, 40, 41, 42] +
        list(range(43, 66)) +
        list(range(67, 93)) +
        list(range(93, 101))
    )

    df = df.drop(columns=drop_cols)

    df['HomeTeamWin'] = (df['HomeScore'] > df['AwayScore']).astype(int)
    df['AwayTeamWin'] = (df['AwayScore'] > df['HomeScore']).astype(int)

    return df


In [ ]:
boxData18 = clean_data_box(path18)
boxData19 = clean_data_box(path19)
boxData20 = clean_data_box(path20)
boxData21 = clean_data_box(path21)
boxData22 = clean_data_box(path22)
boxData23 = clean_data_box(path23)
boxData24 = clean_data_box(path24)
boxData25 = clean_data_box(path25)

In [ ]:
# This function uses Pybaseball to fill in various player IDs for all lineup players and pitchers

def addIDs(df, fetchList):
    
    for col in fetchList:
        df[col + "MLB"] = ""
        df[col + "BRef"] = ""
        df[col + "FG"] = ""
    
    for index, row in df.iterrows():
        
        for player in fetchList:
            
            id = df.loc[index, player]
            temp = pb.playerid_reverse_lookup([id], key_type='retro')
            df.loc[index, player + "MLB"] = str(temp.loc[0,'key_mlbam'])
            df.loc[index, player + "BRef"] = str(temp.loc[0,"key_bbref"])
            df.loc[index, player + "FG"] = str(temp.loc[0,"key_fangraphs"])
        
    return df

toFetch = ['AwayPitcherID', 'HomePitcherID', 'AwayBatter1ID', 'AwayBatter2ID', 'AwayBatter3ID', 'AwayBatter4ID', 'AwayBatter5ID', 'AwayBatter6ID', 'AwayBatter7ID',
           'AwayBatter8ID', 'AwayBatter9ID', 'HomeBatter1ID', 'HomeBatter2ID', 'HomeBatter3ID', 'HomeBatter4ID', 'HomeBatter5ID', 'HomeBatter6ID', 'HomeBatter7ID',
           'HomeBatter8ID', 'HomeBatter9ID']

In [ ]:
boxData18 = addIDs(boxData18, toFetch)
boxData19 = addIDs(boxData19, toFetch)
boxData20 = addIDs(boxData20, toFetch)
boxData21 = addIDs(boxData21, toFetch)
boxData22 = addIDs(boxData22, toFetch)
boxData23 = addIDs(boxData23, toFetch)
boxData24 = addIDs(boxData24, toFetch)
boxData25 = addIDs(boxData25, toFetch)

In [ ]:
# External table that features many of my missing IDs

id_map = pd.read_csv('/Users/owendrummond/Downloads/SFBB Player ID Map - PLAYERIDMAP.csv')
id_map

In [ ]:
# Function to fill some remaining missing IDs and returning list of remaining missing IDs which I will fill manually.

def add_missing_ids(games_df, map):
    
    fix = []
    
    fg_ids = ['AwayPitcherIDFG', 'HomePitcherIDFG', 'AwayBatter1IDFG', 'AwayBatter2IDFG', 'AwayBatter3IDFG', 'AwayBatter4IDFG', 'AwayBatter5IDFG', 'AwayBatter6IDFG', 'AwayBatter7IDFG',
          'AwayBatter8IDFG', 'AwayBatter9IDFG', 'HomeBatter1IDFG', 'HomeBatter2IDFG', 'HomeBatter3IDFG', 'HomeBatter4IDFG', 'HomeBatter5IDFG', 'HomeBatter6IDFG', 'HomeBatter7IDFG',
          'HomeBatter8IDFG', 'HomeBatter9IDFG']
    bref_ids = ['AwayPitcherIDBRef', 'HomePitcherIDBRef', 'AwayBatter1IDBRef', 'AwayBatter2IDBRef', 'AwayBatter3IDBRef', 'AwayBatter4IDBRef', 'AwayBatter5IDBRef', 'AwayBatter6IDBRef', 'AwayBatter7IDBRef',
          'AwayBatter8IDBRef', 'AwayBatter9IDBRef', 'HomeBatter1IDBRef', 'HomeBatter2IDBRef', 'HomeBatter3IDBRef', 'HomeBatter4IDBRef', 'HomeBatter5IDBRef', 'HomeBatter6IDBRef', 'HomeBatter7IDBRef',
          'HomeBatter8IDBRef', 'HomeBatter9IDBRef']
    
    for index, row in games_df.iterrows():
          for fg_player, bref_player in zip(fg_ids, bref_ids):
                if games_df.loc[index, fg_player] == '-1':
                      bref_id = games_df.loc[index, bref_player]
                      #fix.append(bref_id)
                      match = map.loc[map['BREFID'] == bref_id, 'IDFANGRAPHS']
                      
                      if not match.empty:
                        games_df.loc[index, fg_player] = match.iloc[0]
                      else:
                        fix.append(bref_id)
      
    fix = list(set(fix))  
    return games_df, fix

In [ ]:
BoxData18,missing18 = add_missing_ids(boxData18,id_map)
BoxData19,missing19 = add_missing_ids(boxData19,id_map)
BoxData20,missing20 = add_missing_ids(boxData20,id_map)
BoxData21,missing21 = add_missing_ids(boxData21,id_map)
BoxData22,missing22 = add_missing_ids(boxData22,id_map)
BoxData23,missing23 = add_missing_ids(boxData23,id_map)
BoxData24,missing24 = add_missing_ids(boxData24,id_map)
BoxData25,missing25 = add_missing_ids(boxData25,id_map)

In [ ]:
# Validating remaining missing ID counts for FanGraphs IDs

missingIDS = missing18 + missing19 + missing20 + missing21 + missing22 + missing23 + missing24 + missing25
missingIDS = list(set(missingIDS))
print(missingIDS)

In [ ]:
# Extracting first and last names for remaining missing players

missing_first_names= []
missing_last_names= []
for refID in missingIDS:
    lookup = pb.playerid_reverse_lookup([refID], key_type='bbref')
    first_name = lookup['name_first'].iloc[0]
    last_name = lookup['name_last'].iloc[0]
    missing_first_names.append(first_name)
    missing_last_names.append(last_name)

In [ ]:
# Exporting dataframe of missing IDs

manual_missing_players_df = pd.DataFrame({
    'firstName': missing_first_names,
    'lastName': missing_last_names,
    'BRefID': missingIDS
})

manual_missing_players_df.to_csv("missing_players.csv", index=False)

In [ ]:
# After filling remaining IDs manually, load back in the data set.

id_map1 = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/missing_players.csv')
id_map1['IDFANGRAPHS'] = id_map1['IDFANGRAPHS'].astype(str)

In [ ]:
BoxScores18,missing18 = add_missing_ids(BoxData18,id_map1)
BoxScores19,missing19 = add_missing_ids(BoxData19,id_map1)
BoxScores20,missing20 = add_missing_ids(BoxData20,id_map1)
BoxScores21,missing21 = add_missing_ids(BoxData21,id_map1)
BoxScores22,missing22 = add_missing_ids(BoxData22,id_map1)
BoxScores23,missing23 = add_missing_ids(BoxData23,id_map1)
BoxScores24,missing24 = add_missing_ids(BoxData24,id_map1)
BoxScores25,missing25 = add_missing_ids(BoxData25,id_map1)

In [ ]:
# Function to add Year field to data frame and add 'FirstYear' indicator for each player.

def dates_rookies(df, fetch_list):
    df['Year'] = df['Date'].astype(str).str[:4].astype('float64')
    df['Date'] = pd.to_datetime(df['Date'].astype(str), format= '%Y%m%d')
    
    for index, row in df.iterrows():
        
        retroid_lineups = []
        lineup_index = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]
        
        for player in fetch_list:
            retroid_lineups.append(df.loc[index, player])
        
        rookie_map = pb.playerid_reverse_lookup(retroid_lineups, key_type='retro') 
        
        rookie_map = (
            rookie_map
            .set_index('key_retro')
            .loc[retroid_lineups]
            .reset_index()
        )
        
           
        for index_lineup, player1 in zip(lineup_index, fetch_list):
            
            if df.loc[index, 'Year'] == rookie_map.loc[index_lineup, 'mlb_played_first']:
                df.loc[index, player1 + 'FirstYear'] = 1
            else:
                df.loc[index, player1 + 'FirstYear'] = 0
                
    return df
    

In [ ]:
GameBox18 = dates_rookies(BoxScores18,toFetch)
GameBox19 = dates_rookies(BoxScores19,toFetch)
GameBox20 = dates_rookies(BoxScores20,toFetch)
GameBox21 = dates_rookies(BoxScores21,toFetch)
GameBox22 = dates_rookies(BoxScores22,toFetch)
GameBox23 = dates_rookies(BoxScores23,toFetch)
GameBox24 = dates_rookies(BoxScores24,toFetch)
GameBox25 = dates_rookies(BoxScores25,toFetch)

In [ ]:
# Combining years into one data frame and saving an export

GameBoxScores = pd.concat([GameBox18,GameBox19,GameBox20,GameBox21,GameBox22,GameBox23,GameBox24,GameBox25],ignore_index=True)
GameBoxScores.to_csv('GameBoxScores.csv')

In [182]:
GameBoxScores = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/GameBoxScores.csv')

/var/folders/4k/3pfkn1jn039bp8wy8410pn3m0000gn/T/ipykernel_85200/3065319637.py:1: DtypeWarning: Columns (0: AwayPitcherIDFG, 1: HomePitcherIDFG, 2: AwayBatter8IDFG, 3: AwayBatter9IDFG, 4: HomeBatter8IDFG, 5: HomeBatter9IDFG) have mixed types. Specify dtype option on import or set low_memory=False.
  GameBoxScores = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/GameBoxScores.csv')


In [183]:
# Using unique combinations of player IDs, creating the player_dim table which will serve as the source of truth of player names and IDs.

player_dim = duckdb.query("""
    with cte_id as(
    select distinct AwayPitcher as player_name, AwayPitcherID as retro_id, AwayPitcherIDMLB as mlb_id, AwayPitcherIDBRef as bref_id, AwayPitcherIDFG as fg_id
    FROM GameBoxScores
    union
    select distinct HomePitcher as player_name, HomePitcherID as retro_id, HomePitcherIDMLB as mlb_id, HomePitcherIDBRef as bref_id, HomePitcherIDFG as fg_id
    FROM GameBoxScores
    union
    select distinct AwayBatter1 as player_name, AwayBatter1ID as retro_id, AwayBatter1IDMLB as mlb_id, AwayBatter1IDBRef as bref_id, AwayBatter1IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct AwayBatter2 as player_name, AwayBatter2ID as retro_id, AwayBatter2IDMLB as mlb_id, AwayBatter2IDBRef as bref_id, AwayBatter2IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct AwayBatter3 as player_name, AwayBatter3ID as retro_id, AwayBatter3IDMLB as mlb_id, AwayBatter3IDBRef as bref_id, AwayBatter3IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct AwayBatter4 as player_name, AwayBatter4ID as retro_id, AwayBatter4IDMLB as mlb_id, AwayBatter4IDBRef as bref_id, AwayBatter4IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct AwayBatter5 as player_name, AwayBatter5ID as retro_id, AwayBatter5IDMLB as mlb_id, AwayBatter5IDBRef as bref_id, AwayBatter5IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct AwayBatter6 as player_name, AwayBatter6ID as retro_id, AwayBatter6IDMLB as mlb_id, AwayBatter6IDBRef as bref_id, AwayBatter6IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct AwayBatter7 as player_name, AwayBatter7ID as retro_id, AwayBatter7IDMLB as mlb_id, AwayBatter7IDBRef as bref_id, AwayBatter7IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct AwayBatter8 as player_name, AwayBatter8ID as retro_id, AwayBatter8IDMLB as mlb_id, AwayBatter8IDBRef as bref_id, AwayBatter8IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct AwayBatter9 as player_name, AwayBatter9ID as retro_id, AwayBatter9IDMLB as mlb_id, AwayBatter9IDBRef as bref_id, AwayBatter9IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct HomeBatter1 as player_name, HomeBatter1ID as retro_id, HomeBatter1IDMLB as mlb_id, HomeBatter1IDBRef as bref_id, HomeBatter1IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct HomeBatter2 as player_name, HomeBatter2ID as retro_id, HomeBatter2IDMLB as mlb_id, HomeBatter2IDBRef as bref_id, HomeBatter2IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct HomeBatter3 as player_name, HomeBatter3ID as retro_id, HomeBatter3IDMLB as mlb_id, HomeBatter3IDBRef as bref_id, HomeBatter3IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct HomeBatter4 as player_name, HomeBatter4ID as retro_id, HomeBatter4IDMLB as mlb_id, HomeBatter4IDBRef as bref_id, HomeBatter4IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct HomeBatter5 as player_name, HomeBatter5ID as retro_id, HomeBatter5IDMLB as mlb_id, HomeBatter5IDBRef as bref_id, HomeBatter5IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct HomeBatter6 as player_name, HomeBatter6ID as retro_id, HomeBatter6IDMLB as mlb_id, HomeBatter6IDBRef as bref_id, HomeBatter6IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct HomeBatter7 as player_name, HomeBatter7ID as retro_id, HomeBatter7IDMLB as mlb_id, HomeBatter7IDBRef as bref_id, HomeBatter7IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct HomeBatter8 as player_name, HomeBatter8ID as retro_id, HomeBatter8IDMLB as mlb_id, HomeBatter8IDBRef as bref_id, HomeBatter8IDFG as fg_id
    FROM GameBoxScores
    union
    select distinct HomeBatter9 as player_name, HomeBatter9ID as retro_id, HomeBatter9IDMLB as mlb_id, HomeBatter9IDBRef as bref_id, HomeBatter9IDFG as fg_id
    FROM GameBoxScores
)
select distinct player_name, retro_id, mlb_id, bref_id, fg_id
FROM cte_id
""").df()

# Load Data with Statistics
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------
---------------------------------------

### Defense

In [2]:
# Function that takes in the folder path holding defensive CSV files and reads them into a combined data frame.

import glob
import os
import re

def load_fielding_files(folder_path):
    
    # Find all csv files
    files = glob.glob(os.path.join(folder_path, "*.csv"))
    
    all_dfs = []
    
    for file in files:
        
        # Extract filename only
        filename = os.path.basename(file)
        
        # Extract MMYY using regex
        # looks for 4 digits before .csv
        match = re.search(r'(\d{4})(?=\.csv)', filename)
        
        if match:
            mmyy = match.group(1)
            month = mmyy[:2]
            year = "20" + mmyy[2:]   # assumes 2018–2025 like you said
        else:
            month = None
            year = None
        
        # Load CSV
        df = pd.read_csv(file)
        
        # Add columns
        df["Month"] = month
        df["Year"] = year
        
        all_dfs.append(df)
    
    # Union all files
    combined_df = pd.concat(all_dfs, ignore_index=True)
    
    return combined_df



In [3]:
defense_folder_path = '/Users/owendrummond/Downloads/defense_csv_files'
frv_map = load_fielding_files(defense_folder_path)

In [4]:
oaa_folder_path = '/Users/owendrummond/Downloads/oaa_csv_files'
oaa_map = load_fielding_files(oaa_folder_path)

In [5]:
frv_map['innings'] =  frv_map['outs_total'] / 3

In [6]:
# Innings were not included in the 2017 defense data so I loaded in an inning map separately.

inn_map = pd.read_csv('/Users/owendrummond/Downloads/innings_played_2017.csv',header=1)
inn_map

,Rk,Player,Age,Team,Lg,G,GS,CG,Inn,Ch,...,lgRFG,PB,WP,SB,CS,CS%,Pick,Pos,Awards,-9999
0,1,Alcides Escobar,30,KCR,AL,162,162,150,1404.1,685,...,4.04,NaN,NaN,NaN,NaN,NaN,NaN,*6,NaN,escobal02
1,2,Jonathan Schoop,25,BAL,AL,160,160,142,1404.0,805,...,4.29,NaN,NaN,NaN,NaN,NaN,NaN,*4/6,ASMVP-12,schoojo01
2,3,Joey Votto,33,CIN,NL,162,162,146,1391.2,1396,...,8.84,NaN,NaN,NaN,NaN,NaN,NaN,*3,ASMVP-2,vottojo01
3,4,Mookie Betts,24,BOS,AL,153,153,148,1389.1,379,...,2.14,NaN,NaN,NaN,NaN,NaN,NaN,*9,ASMVP-6GG,bettsmo01
4,5,Francisco Lindor,23,CLE,AL,158,158,146,1377.0,611,...,4.04,NaN,NaN,NaN,NaN,NaN,NaN,*6/D,ASMVP-5SS,lindofr01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1616,1350,Roenis Elías,28,BOS,AL,1,0,0,0.1,0,...,1.30,NaN,NaN,NaN,NaN,NaN,NaN,/1,NaN,eliasro01
1617,1351,Damien Magnifico,26,LAA,AL,1,0,0,0.1,0,...,1.30,NaN,NaN,NaN,NaN,NaN,NaN,/1,NaN,magnida01
1618,1352,Hunter Wood,23,TBR,AL,1,0,0,0.1,0,...,1.30,NaN,NaN,NaN,NaN,NaN,NaN,/1,NaN,woodhu01
1619,1353,Jake Esch,27,SDP,NL,1,0,0,0.0,0,...,1.70,NaN,NaN,NaN,NaN,NaN,NaN,/1,NaN,eschja01


In [7]:
inn_map = inn_map.rename(columns={'-9999': 'player_id'})

In [8]:
# 1. Sort by Innings descending so the highest value is at the top
inn_map = inn_map.sort_values('Inn', ascending=False)

# 2. Drop duplicates based on player_id, keeping only the first (highest) occurrence
inn_map = inn_map.drop_duplicates(subset='player_id', keep='first')

In [9]:
# Replaces '.1' at the end of the string with '.33' and '.2' with '.67'
inn_map['Inn'] = (inn_map['Inn'].astype(str)
             .str.replace(r'\.1$', '.33', regex=True)
             .str.replace(r'\.2$', '.67', regex=True)
             .astype(float))

In [10]:
cda_2017 = pd.read_csv('/Users/owendrummond/Downloads/bp_export_20260224.csv')

In [11]:
# Analyzing month field structure for Baseball Prospectus data loads.

cda_clean = duckdb.query("""
        select distinct Month
        from cda_2017
""").df()
cda_clean

,Month
0,July
1,Mar/Apr
2,August
3,June
4,May
5,Sep/Oct


In [12]:
# 2017 catcher defense table with cleaned month values.

cda_clean = duckdb.query("""
        select
        bpid,
        mlbid,
        Name,
        Season,
        CASE
            WHEN "Month" = 'Mar/Apr' THEN '04'
            WHEN "Month" LIKE '%May%' THEN '05'
            WHEN "Month" LIKE '%June%' THEN '06' 
            WHEN "Month" LIKE '%July%' THEN '07' 
            WHEN "Month" LIKE '%August%' THEN '08' 
            WHEN "Month" = 'Sep/Oct' THEN '09' 
        END AS Month,
        CDA
        from cda_2017
""").df()
cda_clean

,bpid,mlbid,Name,Season,Month,CDA
0,102110,620443,Luis Torrens,2017,08,-1.1
1,102110,620443,Luis Torrens,2017,07,-1.5
2,102110,620443,Luis Torrens,2017,06,-1.1
3,102110,620443,Luis Torrens,2017,04,-0.3
4,102110,620443,Luis Torrens,2017,05,-2.4
...,...,...,...,...,...,...
465,42127,434563,Carlos Ruiz,2017,07,-2.7
466,42127,434563,Carlos Ruiz,2017,06,-0.6
467,42127,434563,Carlos Ruiz,2017,04,-2.3
468,42127,434563,Carlos Ruiz,2017,05,-2.7


In [13]:
# Combining catcher and other fielder defensive statistics for 2017.

defense_2017 = duckdb.query("""
        with oaa_cda as(
            select
            player_id as player_id,
            Month as Month,
            year as Year,
            fielding_runs_prevented as fielding_runs
            from oaa_map
            
            union all
            
            select
            mlbid as player_id,
            Month as Month,
            Season as Year,
            CDA as fielding_runs
            from cda_clean
        )
        
        select
        player_id,
        Month,
        Year,
        sum(fielding_runs) as fielding_runs
        from oaa_cda
        group by player_id, Month, Year
        
""").df()
defense_2017

,player_id,Month,Year,fielding_runs
0,547989,09,2017,1.0
1,493114,09,2017,-2.0
2,641319,09,2017,-2.0
3,605137,09,2017,-1.0
4,453568,09,2017,-1.0
...,...,...,...,...
2370,431145,06,2017,-1.5
2371,455755,06,2017,-1.1
2372,452672,07,2017,-2.8
2373,454560,04,2017,-0.7


In [14]:
player_dim = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/player_dim.csv')

In [15]:
# Combining 2017 defense data with the exisitng FRV table. Innings played per month calculation included for 2017 data.

defense_fact = duckdb.query("""
with months_played as(
    select player_id, count(*) as month_count from defense_2017
    group by player_id
),

inn_per_month as(
    select
    mp.player_id as mlb_id,
    im.player_id as bref_id,
    im.Inn / mp.month_count as inn_per_month
    from
    months_played mp
    inner join player_dim pd on mp.player_id = pd.mlb_id
    inner join inn_map im on im.player_id = pd.bref_id
),

defense_cte as(
    select
    om.player_id as mlb_id,
    om.fielding_runs as fielding_runs,
    om.Month as Month,
    om.Year as Year,
    ipm.inn_per_month as innings
    from defense_2017 om
    inner join inn_per_month ipm on om.player_id = ipm.mlb_id

    union all

    select
    fm.id as mlb_id,
    fm.total_runs as fielding_runs,
    fm.Month as Month,
    fm.Year as Year,
    fm.innings as innings
    from frv_map fm
)

select *,
date_trunc('Month', make_date(CAST(Year AS INTEGER), CAST(Month AS INTEGER), 1)) + INTERVAL 1 MONTH - INTERVAL 1 DAY AS CompDate
from defense_cte

""").df()

defense_fact

,mlb_id,fielding_runs,Month,Year,innings,CompDate
0,425772,0.900000,08,2017,79.111667,2017-08-31
1,425772,0.500000,06,2017,79.111667,2017-06-30
2,425772,0.300000,09,2017,79.111667,2017-09-30
3,455104,0.100000,09,2017,105.611667,2017-09-30
4,455755,0.200000,04,2017,59.111667,2017-04-30
...,...,...,...,...,...,...
22874,660162,-4.439094,09,2018,211.333333,2018-09-30
22875,641531,-4.682033,09,2018,185.666667,2018-09-30
22876,622168,-4.726737,09,2018,164.000000,2018-09-30
22877,595777,-4.759406,09,2018,210.333333,2018-09-30


In [18]:
# I realized a mistake of not including catcher defense for 2017 in my initial run of defense_fact

pb.playerid_reverse_lookup([592663], key_type='mlbam')

Gathering player lookup table. This may take a moment.


,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last
0,realmuto,j. t.,592663,realj001,realmjt01,11739,2014.0,2025.0


In [19]:
defense_fact

,mlb_id,fielding_runs,Month,Year,innings,CompDate
0,453568,0.000000,07,2017,227.721667,2017-07-31
1,460075,-1.000000,07,2017,199.582500,2017-07-31
2,608324,0.000000,07,2017,219.388333,2017-07-31
3,595879,3.000000,07,2017,187.166667,2017-07-31
4,466320,-3.000000,07,2017,204.445000,2017-07-31
...,...,...,...,...,...,...
22874,660162,-4.439094,09,2018,211.333333,2018-09-30
22875,641531,-4.682033,09,2018,185.666667,2018-09-30
22876,622168,-4.726737,09,2018,164.000000,2018-09-30
22877,595777,-4.759406,09,2018,210.333333,2018-09-30


In [20]:
# Adding positional ID fields for defensive statistic joining logic.

GameBoxScores = duckdb.query("""
    SELECT *,
    CASE
       WHEN AwayBatter1Pos = 2 THEN AwayBatter1IDMLB
       WHEN AwayBatter2Pos = 2 THEN AwayBatter2IDMLB
       WHEN AwayBatter3Pos = 2 THEN AwayBatter3IDMLB
       WHEN AwayBatter4Pos = 2 THEN AwayBatter4IDMLB
       WHEN AwayBatter5Pos = 2 THEN AwayBatter5IDMLB
       WHEN AwayBatter6Pos = 2 THEN AwayBatter6IDMLB
       WHEN AwayBatter7Pos = 2 THEN AwayBatter7IDMLB
       WHEN AwayBatter8Pos = 2 THEN AwayBatter8IDMLB
       WHEN AwayBatter9Pos = 2 THEN AwayBatter9IDMLB
   END AS AwayCIDMLB,
   CASE
       WHEN AwayBatter1Pos = 3 THEN AwayBatter1IDMLB
       WHEN AwayBatter2Pos = 3 THEN AwayBatter2IDMLB
       WHEN AwayBatter3Pos = 3 THEN AwayBatter3IDMLB
       WHEN AwayBatter4Pos = 3 THEN AwayBatter4IDMLB
       WHEN AwayBatter5Pos = 3 THEN AwayBatter5IDMLB
       WHEN AwayBatter6Pos = 3 THEN AwayBatter6IDMLB
       WHEN AwayBatter7Pos = 3 THEN AwayBatter7IDMLB
       WHEN AwayBatter8Pos = 3 THEN AwayBatter8IDMLB
       WHEN AwayBatter9Pos = 3 THEN AwayBatter9IDMLB
   END AS Away1BIDMLB,
   CASE
       WHEN AwayBatter1Pos = 4 THEN AwayBatter1IDMLB
       WHEN AwayBatter2Pos = 4 THEN AwayBatter2IDMLB
       WHEN AwayBatter3Pos = 4 THEN AwayBatter3IDMLB
       WHEN AwayBatter4Pos = 4 THEN AwayBatter4IDMLB
       WHEN AwayBatter5Pos = 4 THEN AwayBatter5IDMLB
       WHEN AwayBatter6Pos = 4 THEN AwayBatter6IDMLB
       WHEN AwayBatter7Pos = 4 THEN AwayBatter7IDMLB
       WHEN AwayBatter8Pos = 4 THEN AwayBatter8IDMLB
       WHEN AwayBatter9Pos = 4 THEN AwayBatter9IDMLB
   END AS Away2BIDMLB,
   CASE
       WHEN AwayBatter1Pos = 5 THEN AwayBatter1IDMLB
       WHEN AwayBatter2Pos = 5 THEN AwayBatter2IDMLB
       WHEN AwayBatter3Pos = 5 THEN AwayBatter3IDMLB
       WHEN AwayBatter4Pos = 5 THEN AwayBatter4IDMLB
       WHEN AwayBatter5Pos = 5 THEN AwayBatter5IDMLB
       WHEN AwayBatter6Pos = 5 THEN AwayBatter6IDMLB
       WHEN AwayBatter7Pos = 5 THEN AwayBatter7IDMLB
       WHEN AwayBatter8Pos = 5 THEN AwayBatter8IDMLB
       WHEN AwayBatter9Pos = 5 THEN AwayBatter9IDMLB
   END AS Away3BIDMLB,
   CASE
       WHEN AwayBatter1Pos = 6 THEN AwayBatter1IDMLB
       WHEN AwayBatter2Pos = 6 THEN AwayBatter2IDMLB
       WHEN AwayBatter3Pos = 6 THEN AwayBatter3IDMLB
       WHEN AwayBatter4Pos = 6 THEN AwayBatter4IDMLB
       WHEN AwayBatter5Pos = 6 THEN AwayBatter5IDMLB
       WHEN AwayBatter6Pos = 6 THEN AwayBatter6IDMLB
       WHEN AwayBatter7Pos = 6 THEN AwayBatter7IDMLB
       WHEN AwayBatter8Pos = 6 THEN AwayBatter8IDMLB
       WHEN AwayBatter9Pos = 6 THEN AwayBatter9IDMLB
   END AS AwaySSIDMLB,
   CASE
       WHEN AwayBatter1Pos = 7 THEN AwayBatter1IDMLB
       WHEN AwayBatter2Pos = 7 THEN AwayBatter2IDMLB
       WHEN AwayBatter3Pos = 7 THEN AwayBatter3IDMLB
       WHEN AwayBatter4Pos = 7 THEN AwayBatter4IDMLB
       WHEN AwayBatter5Pos = 7 THEN AwayBatter5IDMLB
       WHEN AwayBatter6Pos = 7 THEN AwayBatter6IDMLB
       WHEN AwayBatter7Pos = 7 THEN AwayBatter7IDMLB
       WHEN AwayBatter8Pos = 7 THEN AwayBatter8IDMLB
       WHEN AwayBatter9Pos = 7 THEN AwayBatter9IDMLB
   END AS AwayLFIDMLB,
   CASE
       WHEN AwayBatter1Pos = 8 THEN AwayBatter1IDMLB
       WHEN AwayBatter2Pos = 8 THEN AwayBatter2IDMLB
       WHEN AwayBatter3Pos = 8 THEN AwayBatter3IDMLB
       WHEN AwayBatter4Pos = 8 THEN AwayBatter4IDMLB
       WHEN AwayBatter5Pos = 8 THEN AwayBatter5IDMLB
       WHEN AwayBatter6Pos = 8 THEN AwayBatter6IDMLB
       WHEN AwayBatter7Pos = 8 THEN AwayBatter7IDMLB
       WHEN AwayBatter8Pos = 8 THEN AwayBatter8IDMLB
       WHEN AwayBatter9Pos = 8 THEN AwayBatter9IDMLB
   END AS AwayCFIDMLB,
   CASE
       WHEN AwayBatter1Pos = 9 THEN AwayBatter1IDMLB
       WHEN AwayBatter2Pos = 9 THEN AwayBatter2IDMLB
       WHEN AwayBatter3Pos = 9 THEN AwayBatter3IDMLB
       WHEN AwayBatter4Pos = 9 THEN AwayBatter4IDMLB
       WHEN AwayBatter5Pos = 9 THEN AwayBatter5IDMLB
       WHEN AwayBatter6Pos = 9 THEN AwayBatter6IDMLB
       WHEN AwayBatter7Pos = 9 THEN AwayBatter7IDMLB
       WHEN AwayBatter8Pos = 9 THEN AwayBatter8IDMLB
       WHEN AwayBatter9Pos = 9 THEN AwayBatter9IDMLB
   END AS AwayRFIDMLB,
   CASE
      WHEN HomeBatter1Pos = 2 THEN HomeBatter1IDMLB
      WHEN HomeBatter2Pos = 2 THEN HomeBatter2IDMLB
      WHEN HomeBatter3Pos = 2 THEN HomeBatter3IDMLB
      WHEN HomeBatter4Pos = 2 THEN HomeBatter4IDMLB
      WHEN HomeBatter5Pos = 2 THEN HomeBatter5IDMLB
      WHEN HomeBatter6Pos = 2 THEN HomeBatter6IDMLB
      WHEN HomeBatter7Pos = 2 THEN HomeBatter7IDMLB
      WHEN HomeBatter8Pos = 2 THEN HomeBatter8IDMLB
      WHEN HomeBatter9Pos = 2 THEN HomeBatter9IDMLB
  END AS HomeCIDMLB,
  CASE
      WHEN HomeBatter1Pos = 3 THEN HomeBatter1IDMLB
      WHEN HomeBatter2Pos = 3 THEN HomeBatter2IDMLB
      WHEN HomeBatter3Pos = 3 THEN HomeBatter3IDMLB
      WHEN HomeBatter4Pos = 3 THEN HomeBatter4IDMLB
      WHEN HomeBatter5Pos = 3 THEN HomeBatter5IDMLB
      WHEN HomeBatter6Pos = 3 THEN HomeBatter6IDMLB
      WHEN HomeBatter7Pos = 3 THEN HomeBatter7IDMLB
      WHEN HomeBatter8Pos = 3 THEN HomeBatter8IDMLB
      WHEN HomeBatter9Pos = 3 THEN HomeBatter9IDMLB
  END AS Home1BIDMLB,
  CASE
      WHEN HomeBatter1Pos = 4 THEN HomeBatter1IDMLB
      WHEN HomeBatter2Pos = 4 THEN HomeBatter2IDMLB
      WHEN HomeBatter3Pos = 4 THEN HomeBatter3IDMLB
      WHEN HomeBatter4Pos = 4 THEN HomeBatter4IDMLB
      WHEN HomeBatter5Pos = 4 THEN HomeBatter5IDMLB
      WHEN HomeBatter6Pos = 4 THEN HomeBatter6IDMLB
      WHEN HomeBatter7Pos = 4 THEN HomeBatter7IDMLB
      WHEN HomeBatter8Pos = 4 THEN HomeBatter8IDMLB
      WHEN HomeBatter9Pos = 4 THEN HomeBatter9IDMLB
  END AS Home2BIDMLB,
  CASE
      WHEN HomeBatter1Pos = 5 THEN HomeBatter1IDMLB
      WHEN HomeBatter2Pos = 5 THEN HomeBatter2IDMLB
      WHEN HomeBatter3Pos = 5 THEN HomeBatter3IDMLB
      WHEN HomeBatter4Pos = 5 THEN HomeBatter4IDMLB
      WHEN HomeBatter5Pos = 5 THEN HomeBatter5IDMLB
      WHEN HomeBatter6Pos = 5 THEN HomeBatter6IDMLB
      WHEN HomeBatter7Pos = 5 THEN HomeBatter7IDMLB
      WHEN HomeBatter8Pos = 5 THEN HomeBatter8IDMLB
      WHEN HomeBatter9Pos = 5 THEN HomeBatter9IDMLB
  END AS Home3BIDMLB,
  CASE
      WHEN HomeBatter1Pos = 6 THEN HomeBatter1IDMLB
      WHEN HomeBatter2Pos = 6 THEN HomeBatter2IDMLB
      WHEN HomeBatter3Pos = 6 THEN HomeBatter3IDMLB
      WHEN HomeBatter4Pos = 6 THEN HomeBatter4IDMLB
      WHEN HomeBatter5Pos = 6 THEN HomeBatter5IDMLB
      WHEN HomeBatter6Pos = 6 THEN HomeBatter6IDMLB
      WHEN HomeBatter7Pos = 6 THEN HomeBatter7IDMLB
      WHEN HomeBatter8Pos = 6 THEN HomeBatter8IDMLB
      WHEN HomeBatter9Pos = 6 THEN HomeBatter9IDMLB
  END AS HomeSSIDMLB,
  CASE
      WHEN HomeBatter1Pos = 7 THEN HomeBatter1IDMLB
      WHEN HomeBatter2Pos = 7 THEN HomeBatter2IDMLB
      WHEN HomeBatter3Pos = 7 THEN HomeBatter3IDMLB
      WHEN HomeBatter4Pos = 7 THEN HomeBatter4IDMLB
      WHEN HomeBatter5Pos = 7 THEN HomeBatter5IDMLB
      WHEN HomeBatter6Pos = 7 THEN HomeBatter6IDMLB
      WHEN HomeBatter7Pos = 7 THEN HomeBatter7IDMLB
      WHEN HomeBatter8Pos = 7 THEN HomeBatter8IDMLB
      WHEN HomeBatter9Pos = 7 THEN HomeBatter9IDMLB
  END AS HomeLFIDMLB,
  CASE
      WHEN HomeBatter1Pos = 8 THEN HomeBatter1IDMLB
      WHEN HomeBatter2Pos = 8 THEN HomeBatter2IDMLB
      WHEN HomeBatter3Pos = 8 THEN HomeBatter3IDMLB
      WHEN HomeBatter4Pos = 8 THEN HomeBatter4IDMLB
      WHEN HomeBatter5Pos = 8 THEN HomeBatter5IDMLB
      WHEN HomeBatter6Pos = 8 THEN HomeBatter6IDMLB
      WHEN HomeBatter7Pos = 8 THEN HomeBatter7IDMLB
      WHEN HomeBatter8Pos = 8 THEN HomeBatter8IDMLB
      WHEN HomeBatter9Pos = 8 THEN HomeBatter9IDMLB
  END AS HomeCFIDMLB,
  CASE
      WHEN HomeBatter1Pos = 9 THEN HomeBatter1IDMLB
      WHEN HomeBatter2Pos = 9 THEN HomeBatter2IDMLB
      WHEN HomeBatter3Pos = 9 THEN HomeBatter3IDMLB
      WHEN HomeBatter4Pos = 9 THEN HomeBatter4IDMLB
      WHEN HomeBatter5Pos = 9 THEN HomeBatter5IDMLB
      WHEN HomeBatter6Pos = 9 THEN HomeBatter6IDMLB
      WHEN HomeBatter7Pos = 9 THEN HomeBatter7IDMLB
      WHEN HomeBatter8Pos = 9 THEN HomeBatter8IDMLB
      WHEN HomeBatter9Pos = 9 THEN HomeBatter9IDMLB
  END AS HomeRFIDMLB
   FROM GameBoxScores
""").df()

In [17]:
# Change column type to datetime64[ns] for column: 'CompDate'
defense_fact = defense_fact.astype({'CompDate': 'datetime64[ns]'})

# Change column type to datetime64[ns] for column: 'Date'
#GameBoxScores = GameBoxScores.astype({'Date': 'datetime64[ns]'})

In [18]:
defense_fact.to_csv('defense_fact.csv', index=False)

In [19]:
# Brining in defensive statisitcs over last five months for each player in BoxScores.

GameBoxScores = duckdb.query("""
    SELECT 
GBS.*,
AC.AC_RUNS AS AwayFieldingRunsC,
A1B.A1B_RUNS AS AwayFieldingRuns1B,
A2B.A2B_RUNS AS AwayFieldingRuns2B,
A3B.A3B_RUNS AS AwayFieldingRuns3B,
ASS.ASS_RUNS AS AwayFieldingRunsSS,
ALF.ALF_RUNS AS AwayFieldingRunsLF,
ACF.ACF_RUNS AS AwayFieldingRunsCF,
ARF.ARF_RUNS AS AwayFieldingRunsRF,
HC.HC_RUNS AS HomeFieldingRunsC,
H1B.H1B_RUNS AS HomeFieldingRuns1B,
H2B.H2B_RUNS AS HomeFieldingRuns2B,
H3B.H3B_RUNS AS HomeFieldingRuns3B,
HSS.HSS_RUNS AS HomeFieldingRunsSS,
HLF.HLF_RUNS AS HomeFieldingRunsLF,
HCF.HCF_RUNS AS HomeFieldingRunsCF,
HRF.HRF_RUNS AS HomeFieldingRunsRF

FROM GameBoxScores GBS

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as AC_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.AwayCIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) AC ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as A1B_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.Away1BIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) A1B ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as A2B_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.Away2BIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) A2B ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as A3B_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.Away3BIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) A3B ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as ASS_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.AwaySSIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) ASS ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as ALF_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.AwayLFIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) ALF ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as ACF_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.AwayCFIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) ACF ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as ARF_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.AwayRFIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) ARF ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as HC_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.HomeCIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) HC ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as H1B_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.Home1BIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) H1B ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as H2B_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.Home2BIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) H2B ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as H3B_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.Home3BIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) H3B ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as HSS_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.HomeSSIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) HSS ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as HLF_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.HomeLFIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) HLF ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as HCF_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.HomeCFIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) HCF ON TRUE

LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as HRF_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.HomeRFIDMLB
        AND df.CompDate < GBS.Date
        ORDER BY df.CompDate DESC
        LIMIT 5
    )
) HRF ON TRUE
""").df()

CatalogException: Catalog Error: Table with name GameBoxScores does not exist!
Did you mean "pg_namespace"?

--------------------------
--------------------------
--------------------------

In [32]:
GameBoxScores.to_csv('GameBoxScoresWithDefense.csv')

In [92]:
GameBoxScores = pd.read_csv("/Users/owendrummond/Documents/python_projects/Capstone/GameBoxScoresWithDefense.csv")
GameBoxScores

/var/folders/4k/3pfkn1jn039bp8wy8410pn3m0000gn/T/ipykernel_85200/2674139459.py:1: DtypeWarning: Columns (0: AwayPitcherIDFG, 1: HomePitcherIDFG, 2: AwayBatter8IDFG, 3: AwayBatter9IDFG, 4: HomeBatter8IDFG, 5: HomeBatter9IDFG) have mixed types. Specify dtype option on import or set low_memory=False.
  GameBoxScores = pd.read_csv("/Users/owendrummond/Documents/python_projects/Capstone/GameBoxScoresWithDefense.csv")


,Unnamed: 0.1,Unnamed: 0,Date,DayOfWeek,AwayTeam,AwayLeague,AwayGameNum,HomeTeam,HomeLeague,HomeGameNum,...,HomeFieldingRunsC,HomeFieldingRuns1B,HomeFieldingRuns2B,HomeFieldingRuns3B,HomeFieldingRunsSS,HomeFieldingRunsLF,HomeFieldingRunsCF,HomeFieldingRunsRF,AwayPitcherHand,HomePitcherHand
0,0,9028,2022-06-08,Wed,SEA,AL,57,HOU,AL,57,...,-2.148963,-0.228868,3.797243,0.038807,8.218446,-1.330299,4.548419,1.849546,R,R
1,1,9029,2022-06-08,Wed,TOR,AL,56,KCA,AL,55,...,-6.077389,1.120611,18.819645,-5.068393,0.365701,-2.739635,13.700240,3.259650,L,R
2,2,9030,2022-06-08,Wed,NYA,AL,56,MIN,AL,58,...,6.882673,-1.916596,-1.676249,-8.027219,0.932751,0.631740,6.905421,7.672878,L,R
3,3,9031,2022-06-08,Wed,SLN,NL,57,TBA,AL,56,...,-4.059295,-1.807590,-1.861266,-2.070792,-1.631733,-1.969140,9.153060,1.652700,L,R
4,4,9033,2022-06-09,Thu,ARI,NL,59,CIN,NL,57,...,1.689793,-0.158377,-0.058074,-0.095140,-0.359091,-5.012076,0.543185,1.022030,R,R
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17901,17901,4173,2019-08-10,Sat,CHN,NL,117,CIN,NL,115,...,2.131130,0.666260,0.135987,-0.120783,5.482100,-5.876017,1.697589,NaN,R,R
17902,17902,4176,2019-08-10,Sat,TEX,AL,116,MIL,NL,118,...,15.558143,2.431441,-2.283254,-0.530791,-3.513675,-6.082500,9.873985,NaN,R,R
17903,17903,4184,2019-08-10,Sat,KCA,AL,119,DET,AL,115,...,-0.301219,-0.937077,3.020984,-3.862799,-0.096906,-0.800135,2.018480,NaN,L,R
17904,17904,4188,2019-08-11,Sun,CHN,NL,118,CIN,NL,116,...,2.131130,0.666260,0.286682,-0.120783,5.482100,2.064988,1.697589,NaN,L,R


# Pitching

--------------------
---------------------
--------------------

### Starters

In [10]:
# Extracting unique list of starting pitcher IDs to use in the creation of the fg_pitching_fact table

unique_pitchers = pd.concat([GameBoxScores['AwayPitcherIDMLB'], GameBoxScores['HomePitcherIDMLB']]).unique()
unique_pitchers_list = unique_pitchers.tolist()
print(len(unique_pitchers_list))

1071


In [4]:
# fg_pitching_fact is created using R. The script which generates this CSV file is located in the "projectR.ipynb" file

fangraphs_pitching_fact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/fg_pitching_fact.csv')

fangraphs_pitching_fact['IP'] = (fangraphs_pitching_fact['IP'].astype(str)
             .str.replace(r'\.1$', '.33', regex=True)
             .str.replace(r'\.2$', '.67', regex=True)
             .astype(float))

fangraphs_pitching_fact.to_csv("fangraphs_pitching_fact.csv", index=False)

In [93]:
GameBoxScores = GameBoxScores.astype({'Year': 'int64'})
GameBoxScores = GameBoxScores.rename(columns={'Unnamed: 0.1': 'GAMEID'})

In [94]:
# Load the last two years of starting pitcher data into GameBoxScores

GameBoxScores_w_pitching = duckdb.query("""
WITH stats_weighted AS (
    SELECT
        xMLBAMID,
        Season,
        IP,
        Events,
        WAR,
        (ERA * IP) as ERA_W,
        (K_9 * IP) as K_9_W,
        (BB_9 * IP) as BB_9_W,
        (K_BB * IP) as K_BB_W,
        (FIP * IP) as FIP_W,
        (xFIP * IP) as xFIP_W,
        (SIERA * IP) as SIERA_W,
        (kwERA * IP) as kwERA_W,
        (xERA * IP) as xERA_W,
        (Barrel_pct * Events) as Barrel_pct_W,
        (GB_FB * Events) as GB_FB_W,
        (GB_pct * Events) as GB_pct_W,
        (FB_pct * Events) as FB_pct_W
    FROM fangraphs_pitching_fact
)

select GBS.*,
    -- Home Pitcher L2 Stats
    hP.HomeSP_IP_L2,
    hP.HomeSP_WAR_L2,
    hP.HomeSP_ERA_L2,
    hP.HomeSP_K_9_L2,
    hP.HomeSP_BB_9_L2,
    hP.HomeSP_K_BB_L2,
    hP.HomeSP_FIP_L2,
    hP.HomeSP_xFIP_L2,
    hP.HomeSP_SIERA_L2,
    hP.HomeSP_kwERA_L2,
    hP.HomeSP_xERA_L2,
    hP.HomeSP_Barrel_pct_L2,
    hp.HomeSP_GB_FB_L2,
    hp.HomeSP_GB_pct_L2,
    hp.HomeSP_FB_pct_L2,
    
    -- Away Pitcher L2 Stats
   ap.AwaySP_IP_L2,
   ap.AwaySP_WAR_L2,
   ap.AwaySP_ERA_L2,
   ap.AwaySP_K_9_L2,
   ap.AwaySP_BB_9_L2,
   ap.AwaySP_K_BB_L2,
   ap.AwaySP_FIP_L2,
   ap.AwaySP_xFIP_L2,
   ap.AwaySP_SIERA_L2,
   ap.AwaySP_kwERA_L2,
   ap.AwaySP_xERA_L2,
   ap.AwaySP_Barrel_pct_L2,
   ap.AwaySP_GB_FB_L2,
   ap.AwaySP_GB_pct_L2,
   ap.AwaySP_FB_pct_L2

FROM GameBoxScores GBS

LEFT JOIN LATERAL(
    select 
    SUM(IP) as HomeSP_IP_L2,
    SUM(WAR) as HomeSP_WAR_L2,
    SUM(ERA_W) / NULLIF(SUM(IP), 0) as HomeSP_ERA_L2,
    SUM(K_9_W) / NULLIF(SUM(IP), 0) as HomeSP_K_9_L2,
    SUM(BB_9_W) / NULLIF(SUM(IP), 0) as HomeSP_BB_9_L2,
    SUM(K_BB_W) / NULLIF(SUM(IP), 0) as HomeSP_K_BB_L2,
    SUM(FIP_W) / NULLIF(SUM(IP), 0) as HomeSP_FIP_L2,
    SUM(xFIP_W) / NULLIF(SUM(IP), 0) as HomeSP_xFIP_L2,
    SUM(SIERA_W) / NULLIF(SUM(IP), 0) as HomeSP_SIERA_L2,
    SUM(kwERA_W) / NULLIF(SUM(IP), 0) as HomeSP_kwERA_L2,
    SUM(xERA_W) / NULLIF(SUM(IP), 0) as HomeSP_xERA_L2,
    SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeSP_Barrel_pct_L2,
    SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeSP_GB_FB_L2,
    SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as HomeSP_GB_pct_L2,
    SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as HomeSP_FB_pct_L2
    FROM(
        SELECT
        IP,
        Events,
        WAR,
        ERA_W,
        K_9_W,
        BB_9_W,
        K_BB_W,
        FIP_W,
        xFIP_W,
        SIERA_W,
        kwERA_W,
        xERA_W,
        Barrel_pct_W,
        GB_FB_W,
        GB_pct_W,
        FB_pct_W
        from stats_weighted sw
        where sw.xMLBAMID = GBS.HomePitcherIDMLB
        and sw.Season < GBS.Year
        order by sw.Season DESC
        limit 2
    )
) hp on TRUE

LEFT JOIN LATERAL(
   select
   SUM(IP) as AwaySP_IP_L2,
   SUM(WAR) as AwaySP_WAR_L2,
   SUM(ERA_W) / NULLIF(SUM(IP), 0) as AwaySP_ERA_L2,
   SUM(K_9_W) / NULLIF(SUM(IP), 0) as AwaySP_K_9_L2,
   SUM(BB_9_W) / NULLIF(SUM(IP), 0) as AwaySP_BB_9_L2,
   SUM(K_BB_W) / NULLIF(SUM(IP), 0) as AwaySP_K_BB_L2,
   SUM(FIP_W) / NULLIF(SUM(IP), 0) as AwaySP_FIP_L2,
   SUM(xFIP_W) / NULLIF(SUM(IP), 0) as AwaySP_xFIP_L2,
   SUM(SIERA_W) / NULLIF(SUM(IP), 0) as AwaySP_SIERA_L2,
   SUM(kwERA_W) / NULLIF(SUM(IP), 0) as AwaySP_kwERA_L2,
   SUM(xERA_W) / NULLIF(SUM(IP), 0) as AwaySP_xERA_L2,
   SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwaySP_Barrel_pct_L2,
   SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwaySP_GB_FB_L2,
   SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as AwaySP_GB_pct_L2,
   SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as AwaySP_FB_pct_L2
   FROM(
       SELECT
       IP,
       Events,
       WAR,
       ERA_W,
       K_9_W,
       BB_9_W,
       K_BB_W,
       FIP_W,
       xFIP_W,
       SIERA_W,
       kwERA_W,
       xERA_W,
       Barrel_pct_W,
       GB_FB_W,
       GB_pct_W,
       FB_pct_W
       from stats_weighted sw
       where sw.xMLBAMID = GBS.AwayPitcherIDMLB
       and sw.Season < GBS.Year
       order by sw.Season DESC
       limit 2
   )
) ap on TRUE


""").df()

In [24]:
import mlbstatsapi

mlb = mlbstatsapi.Mlb()

In [28]:
# Use persons objects returned by the mlbstatsapi to get starting pitcher throwing hand.

pitcher_dict = []

for pitcher in unique_pitchers_list:
    pitcher_info = mlb.get_persons([pitcher])
    
    pitcher_dict.append({
        "mlb_id": pitcher_info[0].id,
        "pitch_hand": pitcher_info[0].pitch_hand.code
    })

pitcher_hand_df = pd.DataFrame(pitcher_dict)

In [33]:
pitcher_hand_df.to_csv('pitcher_hand_df.csv')

In [ ]:
# Adding pitcher hand for both away and home pitchers:

In [29]:
GameBoxScores = duckdb.query("""
    
    select GBS.*,
    PH.pitch_hand as AwayPitcherHand
    from
    GameBoxScores GBS
    left join pitcher_hand_df PH on GBS.AwayPitcherIDMLB = PH.mlb_id
                     
 """).df()

In [30]:
GameBoxScores = duckdb.query("""
    
    select GBS.*,
    PH.pitch_hand as HomePitcherHand
    from
    GameBoxScores GBS
    left join pitcher_hand_df PH on GBS.HomePitcherIDMLB = PH.mlb_id
                     
 """).df()

In [46]:
# Pitching stats from baseball prospectus, used for L5M window.

dra17 = pd.read_csv('/Users/owendrummond/Downloads/bp_export_20260418.csv')
dra18 = pd.read_csv('/Users/owendrummond/Downloads/bp_pitching_csv_files/BPPitching2018.csv')
dra19 = pd.read_csv('/Users/owendrummond/Downloads/bp_pitching_csv_files/BPPitching2019.csv')
dra20 = pd.read_csv('/Users/owendrummond/Downloads/bp_pitching_csv_files/BPPitching2020.csv')
dra21 = pd.read_csv('/Users/owendrummond/Downloads/bp_pitching_csv_files/BPPitching2021.csv')
dra22 = pd.read_csv('/Users/owendrummond/Downloads/bp_pitching_csv_files/BPPitching2022.csv')
dra23 = pd.read_csv('/Users/owendrummond/Downloads/bp_pitching_csv_files/BPPitching2023.csv')
dra24 = pd.read_csv('/Users/owendrummond/Downloads/bp_pitching_csv_files/BPPitching2024.csv')
dra25 = pd.read_csv('/Users/owendrummond/Downloads/bp_pitching_csv_files/BPPitching2025.csv')

dra_map = pd.concat([dra17, dra18, dra19, dra20, dra21, dra22, dra23, dra24, dra25])
print(len(dra17) + len(dra18) + len(dra19) + len(dra20) + len(dra21) + len(dra22) + len(dra23) + len(dra24) + len(dra25))

27452


In [47]:
# Cleaning the month field and adding a comparison date for baseball prospectus pitching stats.

dra_map_clean = duckdb.query("""
        with sq as(
        select 
        bpid,
        mlbid,
        Name,
        Season as Year,
        CASE
            WHEN "Month" = 'Mar/Apr' THEN '04'
            WHEN "Month" LIKE '%May%' THEN '05'
            WHEN "Month" LIKE '%June%' THEN '06' 
            WHEN "Month" LIKE '%July%' THEN '07' 
            WHEN "Month" LIKE '%August%' THEN '08' 
            WHEN "Month" = 'Sep/Oct' THEN '09' 
        END AS Month,
        IP,
        ERA,
        DRA,
        "K%",
        "BB%"
        FROM dra_map),
        
        add_date as(
            select *,
            date_trunc('Month', make_date(CAST(Year AS INTEGER), CAST(Month AS INTEGER), 1)) + INTERVAL 1 MONTH - INTERVAL 1 DAY AS CompDate
            from sq
        )
        
        select * from add_date
""").df()

In [ ]:
dra_map_clean

In [48]:
dra_map_clean.to_csv("dra_map_clean.csv", index=False)

In [95]:
# Aggregating baseball prospectus pitching stats, then adding them to GameBoxScores for the last five months.

GameBoxScores_w_pitching1 = duckdb.query("""
with stats_weighted as (
        SELECT
        bpid,
        mlbid,
        Year,
        Month,
        IP,
        CompDate,
        (ERA * IP) as ERA_W,
        (DRA * IP) as DRA_W,
        ("K%" * IP) as Kpct_W,
        ("BB%" * IP) as BBpct_W
    FROM dra_map_clean
)


select GBS.*,
hp.HomeSP_IP_L5M,
hp.HomeSP_ERA_L5M,
hp.HomeSP_DRA_L5M,
hp.HomeSP_Kpct_L5M,
hp.HomeSP_BBpct_L5M,
ap.AwaySP_IP_L5M,
ap.AwaySP_ERA_L5M,
ap.AwaySP_DRA_L5M,
ap.AwaySP_Kpct_L5M,
ap.AwaySP_BBpct_L5M

FROM GameBoxScores_w_pitching GBS

LEFT JOIN LATERAL(
    select 
    SUM(IP) as HomeSP_IP_L5M,
    SUM(ERA_W) / NULLIF(SUM(IP), 0) as HomeSP_ERA_L5M,
    SUM(DRA_W) / NULLIF(SUM(IP), 0) as HomeSP_DRA_L5M,
    SUM(Kpct_W) / NULLIF(SUM(IP), 0) as HomeSP_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(IP), 0) as HomeSP_BBpct_L5M
    FROM(
        SELECT
        IP,
        ERA_W,
        DRA_W,
        Kpct_W,
        BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomePitcherIDMLB
        and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) hp on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(IP) as AwaySP_IP_L5M,
    SUM(ERA_W) / NULLIF(SUM(IP), 0) as AwaySP_ERA_L5M,
    SUM(DRA_W) / NULLIF(SUM(IP), 0) as AwaySP_DRA_L5M,
    SUM(Kpct_W) / NULLIF(SUM(IP), 0) as AwaySP_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(IP), 0) as AwaySP_BBpct_L5M
    FROM(
        SELECT
        IP,
        ERA_W,
        DRA_W,
        Kpct_W,
        BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayPitcherIDMLB
        and CAST(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) ap on TRUE

""").df()

### Bullpen

In [11]:
# Loading baseball reference monthly data for pitchers

bref_monthly_sp = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/bref_monthly_sp.csv')
bref_monthly_rp = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/bref_monthly_rp.csv')

In [ ]:
bref_monthly_rp

In [12]:
cols_to_fix = ['Level', 'Team']

for col in cols_to_fix:
    # 1. Split by comma
    # 2. Take the last element [-1]
    # 3. Strip any leading/trailing whitespace
    bref_monthly_rp[col] = bref_monthly_rp[col].astype(str).str.split(',').str[-1].str.strip()

In [13]:
bref_monthly_rp['Team'].unique()

array(['Tampa Bay', 'San Francisco', 'Toronto', 'Houston', 'Arizona',
       'Milwaukee', 'Detroit', 'New York', 'Pittsburgh', 'Chicago',
       'Los Angeles', 'Cincinnati', 'Oakland', 'Miami', 'Baltimore',
       'Boston', 'Minnesota', 'San Diego', 'Philadelphia', 'Colorado',
       'Kansas City', 'St. Louis', 'Atlanta', 'Seattle', 'Cleveland',
       'Texas', 'Washington', 'Athletics'], dtype=object)

In [14]:
# Team abbreviation standardization for bullpen table.

abbr_conversion = {
    ('Tampa Bay', 'Maj-AL'): 'TBR',
    ('San Francisco', 'Maj-NL'): 'SFG',
    ('Toronto', 'Maj-AL'): 'TOR',
    ('Houston', 'Maj-AL'): 'HOU',
    ('Arizona', 'Maj-NL'): 'ARI',
    ('Milwaukee', 'Maj-NL'): 'MIL',
    ('Detroit', 'Maj-AL'): 'DET',
    ('New York', 'Maj-AL'): 'NYY',
    ('New York', 'Maj-NL'): 'NYM',
    ('Pittsburgh', 'Maj-NL'): 'PIT',
    ('Chicago', 'Maj-AL'): 'CHW',
    ('Chicago', 'Maj-NL'): 'CHC',
    ('Los Angeles', 'Maj-AL'): 'LAA',
    ('Los Angeles', 'Maj-NL'): 'LAD',
    ('Cincinnati', 'Maj-NL'): 'CIN',
    ('Oakland', 'Maj-AL'): 'ATH',
    ('Athletics', 'Maj-AL'): 'ATH',
    ('Miami', 'Maj-NL'): 'MIA',
    ('Baltimore', 'Maj-AL'): 'BAL',
    ('Boston', 'Maj-AL'): 'BOS',
    ('Minnesota', 'Maj-AL'): 'MIN',
    ('San Diego', 'Maj-NL'): 'SDP',
    ('Philadelphia', 'Maj-NL'): 'PHI',
    ('Colorado', 'Maj-NL'): 'COL',
    ('Kansas City', 'Maj-AL'): 'KCR',
    ('St. Louis', 'Maj-NL'): 'STL',
    ('Atlanta', 'Maj-NL'): 'ATL',
    ('Seattle', 'Maj-AL'): 'SEA',
    ('Cleveland', 'Maj-AL'): 'CLE',
    ('Texas', 'Maj-AL'): 'TEX',
    ('Washington', 'Maj-NL'): 'WAS'
}

def get_abbr(row):

    key = (row['Team'], row['Level'])
    
    return abbr_conversion.get(key, row['Team'])


bref_monthly_rp['Team_Abbr'] = bref_monthly_rp.apply(get_abbr, axis=1)

In [15]:
# There were some leftover teams that did not get fixed in the previous script.

additional_abbr_replacements = {
    'Seattle': 'SEA',
    'Tampa Bay': 'TBR',
    'Texas': 'TEX',
    'Oakland': 'ATH',
    'Toronto': 'TOR',
    'Baltimore': 'BAL',
    'Minnesota': 'MIN',
    'Houston': 'HOU',
    'Detroit': 'DET'
}

bref_monthly_rp['Team_Abbr'] = bref_monthly_rp['Team_Abbr'].replace(additional_abbr_replacements)

In [16]:
bref_monthly_rp['Team_Abbr'].unique()

<ArrowStringArray>
['TBR', 'SFG', 'TOR', 'HOU', 'ARI', 'MIL', 'DET', 'NYM', 'PIT', 'CHC', 'LAA',
 'CIN', 'ATH', 'MIA', 'LAD', 'BAL', 'BOS', 'NYY', 'MIN', 'SDP', 'PHI', 'COL',
 'KCR', 'STL', 'CHW', 'ATL', 'SEA', 'CLE', 'TEX', 'WAS']
Length: 30, dtype: str

In [98]:
team_abbrs = GameBoxScores_w_pitching1['HomeTeam'].unique()
team_abbrs

<ArrowStringArray>
['CHC', 'COL', 'WAS', 'BOS', 'CLE', 'DET', 'MIN', 'SEA', 'ATL', 'TOR', 'CIN',
 'LAD', 'MIA', 'STL', 'CHW', 'KCR', 'NYY', 'TBR', 'HOU', 'SDP', 'MIL', 'PHI',
 'PIT', 'SFG', 'LAA', 'TEX', 'ARI', 'BAL', 'NYM', 'ATH']
Length: 30, dtype: str

In [96]:
# Standardizing team abbreviations in GameBoxScores

gbs_abbrs = {
    'KCA': 'KCR',
    'TBA': 'TBR',
    'SFN': 'SFG',
    'ANA': 'LAA',
    'CHA': 'CHW',
    'SDN': 'SDP',
    'SLN': 'STL',
    'NYA': 'NYY',
    'CHN': 'CHC',
    'LAN': 'LAD',
    'NYN': 'NYM',
    'OAK': 'ATH'
}

GameBoxScores_w_pitching1['AwayTeam'] = GameBoxScores_w_pitching1['AwayTeam'].replace(gbs_abbrs)
GameBoxScores_w_pitching1['HomeTeam'] = GameBoxScores_w_pitching1['HomeTeam'].replace(gbs_abbrs)

In [20]:
# Fixing display of innings pitched.

bref_monthly_rp['IP'] = (bref_monthly_rp['IP'].astype(str)
             .str.replace(r'\.1$', '.33', regex=True)
             .str.replace(r'\.2$', '.67', regex=True)
             .astype(float))

True ABBRs

Astros: HOU, Royals: KCR, Twins: MIN, Rays: TBR, Reds: CIN, Marlins: MIA, Brewers: MIL, Gians: SFG, Angels: LAA, White Sox: CHW, Guardians: CLE,
Phillies: PHI, Padres: SDP, Cardinals: STL, Nationals: WAS, Tigers: DET, Yankees: NYY, Mariners: SEA, Braves: ATL, Cubs: CHC, Diamondbacks: ARI, Rockies: COL,
Dodgers: LAD, Mets: NYM, Pirates: PIT, Orioles: BAL, Red Sox: BOS, Blue Jays: TOR, Athletics: ATH, Rangers: TEX

In [25]:
# Aggregating bullpen statistics by team, year, and month/

team_bullpen_monthly = duckdb.query("""
with cte as (
    SELECT
    Team_Abbr,
    query_year,
    query_month,
    SUM(G) as G,
    SUM(GS) as GS,
    SUM(SV) as SV,
    SUM(IP) as IP,
    SUM(H) as H,
    SUM(R) as R,
    SUM(ER) as ER,
    SUM(BB) as BB,
    SUM(SO) as SO,
    SUM(HR) as HR,
    SUM(HBP) as HBP,
    SUM(AB) as AB,
    SUM(IBB) as IBB,
    SUM(GDP) as GDP
FROM bref_monthly_rp
GROUP BY
Team_Abbr,
query_year,
query_month)

select *,
(ER / IP) * 9 as ERA,
(SO / IP) * 9 as SO9,
(BB / IP) * 9 as BB9
from cte
order by query_year, query_month, Team_Abbr
""").df()

In [53]:
team_bullpen_monthly.to_csv("team_bullpen_monthly.csv", index=False)

In [26]:
team_bullpen_monthly['CompDate'] = (
    pd.to_datetime(team_bullpen_monthly['query_year'].astype(str) + '-' + team_bullpen_monthly['query_month'].astype(str) + '-01')
    + pd.offsets.MonthEnd(0)
)

In [ ]:
team_bullpen_monthly

In [99]:
# Adding last three months of bullpen statistics, joining on teams.

GameBoxScores_w_bullpen = duckdb.query("""
with stats_weighted as (
        SELECT
        Team_Abbr,
        query_year,
        query_month,
        IP,
        CompDate,
        (ERA * IP) as ERA_W,
        (SO9 * IP) as SO9_W,
        (BB9 * IP) as BB9_W
    FROM team_bullpen_monthly
)


select GBS.*,
hp.HomeBP_IP_L3M,
hp.HomeBP_ERA_L3M,
hp.HomeBP_SO9_L3M,
hp.HomeBP_BB9_L3M,
ap.AwayBP_IP_L3M,
ap.AwayBP_ERA_L3M,
ap.AwayBP_SO9_L3M,
ap.AwayBP_BB9_L3M

FROM GameBoxScores_w_pitching1 GBS

LEFT JOIN LATERAL(
    select 
    SUM(IP) as HomeBP_IP_L3M,
    SUM(ERA_W) / NULLIF(SUM(IP), 0) as HomeBP_ERA_L3M,
    SUM(SO9_W) / NULLIF(SUM(IP), 0) as HomeBP_SO9_L3M,
    SUM(BB9_W) / NULLIF(SUM(IP), 0) as HomeBP_BB9_L3M
    FROM(
        SELECT
        IP,
        ERA_W,
        SO9_W,
        BB9_W
        from stats_weighted sw
        where sw.Team_Abbr = GBS.HomeTeam
        and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 3
    )
) hp on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(IP) as AwayBP_IP_L3M,
    SUM(ERA_W) / NULLIF(SUM(IP), 0) as AwayBP_ERA_L3M,
    SUM(SO9_W) / NULLIF(SUM(IP), 0) as AwayBP_SO9_L3M,
    SUM(BB9_W) / NULLIF(SUM(IP), 0) as AwayBP_BB9_L3M
    FROM(
        SELECT
        IP,
        ERA_W,
        SO9_W,
        BB9_W
        from stats_weighted sw
        where sw.Team_Abbr = GBS.AwayTeam
        and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 3
    )
) ap on TRUE

""").df()

In [ ]:
GameBoxScores_w_bullpen

In [102]:
# I noticed some incorrectly formatted FanGraphs IDs in the table.

non_numeric_maskH = ~GameBoxScores_w_bullpen['HomePitcherIDFG'].astype(str).str.isnumeric()
non_numeric_pitchersH = GameBoxScores_w_bullpen.loc[non_numeric_maskH, ['HomePitcherIDFG', 'HomePitcher']]

non_numeric_maskA = ~GameBoxScores_w_bullpen['AwayPitcherIDFG'].astype(str).str.isnumeric()
non_numeric_pitchersA = GameBoxScores_w_bullpen.loc[non_numeric_maskA, ['AwayPitcherIDFG', 'AwayPitcher']]

resultsH = non_numeric_pitchersH.drop_duplicates()
resultsA = non_numeric_pitchersA.drop_duplicates()

# Display the results
if not resultsH.empty:
    print(resultsH) 
else:
    print("All HomePitcherIDFG values are numeric.")
    
if not resultsA.empty:
    print(resultsA)
else:
    print("All AwayPitcherIDFG values are numeric.")

All HomePitcherIDFG values are numeric.
All HomePitcherIDFG values are numeric.


Actual Fangraphs IDs To Replace:

Ky Bush(H/A): sa3017386, 29823
Blade Tidwell(A): sa3020177, 31701
Jayden Murray(A): sa1170405, 26482

In [101]:
# Apply manual override of FanGraphs player IDs for specific players.

name_conversion = {
    ('sa3017386', 'Ky Bush'): 29823,
    ('sa3020177', 'Blade Tidwell'): 31701,
    ('sa1170405', 'Jayden Murray'): 26482
}

def get_abbr1(row):
    # Create the lookup key from the two columns
    key = (row['HomePitcherIDFG'], row['HomePitcher'])
    
    # Return the mapped abbreviation, or the original Team name if not found
    return name_conversion.get(key, row['HomePitcherIDFG'])

def get_abbr2(row):
    key = (row['AwayPitcherIDFG'], row['AwayPitcher'])
    
    return name_conversion.get(key, row['AwayPitcherIDFG'])

# Apply the function to create your standardized column
GameBoxScores_w_bullpen['HomePitcherIDFG'] = GameBoxScores_w_bullpen.apply(get_abbr1, axis=1)
GameBoxScores_w_bullpen['AwayPitcherIDFG'] = GameBoxScores_w_bullpen.apply(get_abbr2, axis=1)

In [105]:
GameBoxScores_w_bullpen = GameBoxScores_w_bullpen.replace('sa3008602', 25999)

In [63]:
GameBoxScores_w_bullpen.to_csv("GameBoxScores_PostBullpen.csv", index=False)

## NOTE: At this point, GBS will be exported to R file to load recent start (game-by-game) data.

In [4]:
# This table was created in the "projectR.ipynb" file. It contains all pitcher game logs for each pitcher-year unique combination.

SPGameLogFact = pd.read_csv("/Users/owendrummond/Documents/python_projects/Capstone/SPGameLogFact.csv")
SPGameLogFact26 = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/StartingPGameLogFact2026.csv')
SPGameLogFact

,PlayerName,playerid,Date,Opp,teamid,season,Team,HomeAway,Age,W,...,pfxFO-X,pfxFO-Z,pfxwFO,pfxwFO/C,pfxaaFO,pfxspFO,rDSV,sp_s_FO,sp_l_FO,sp_p_FO
0,Andrew Cashner,8782,2019-09-28,BAL,3,2019,BOS,H,32,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Andrew Cashner,8782,2019-09-27,BAL,3,2019,BOS,H,32,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Andrew Cashner,8782,2019-09-25,@TEX,3,2019,BOS,A,32,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Andrew Cashner,8782,2019-09-24,@TEX,3,2019,BOS,A,32,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Andrew Cashner,8782,2019-09-21,@TBR,3,2019,BOS,A,32,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62823,Jose Ruiz,14552,2023-04-03,SFG,4,2023,CHW,H,28,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62824,Jose Ruiz,14552,2023-04-01,@HOU,4,2023,CHW,A,28,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62825,Jose Ruiz,14552,2023-03-31,@HOU,4,2023,CHW,A,28,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
62826,Janson Junk,23301,2023-10-01,CHC,23,2023,MIL,H,27,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# IGNORE (CONCAT FOR LIVE FEATURE, STILL IN PROGRESS)

common_cols = list(set(SPGameLogFact.columns) & set(SPGameLogFact26.columns))

SP_Combined_Fact = pd.concat([
    SPGameLogFact[common_cols], 
    SPGameLogFact26[common_cols]
], ignore_index=True)

In [8]:
SP_Combined_Fact.to_csv('SPGameLogFactCombined.csv')

In [108]:
# Adding last four starts data for starting pitchers to GameBoxScores.

GameBoxScoresDAP = duckdb.query("""
with stats_weighted as (
    SELECT
        playerid,
        season,
        IP,
        Events,
        Date,
        (ERA * IP) as ERA_W,
        ("K/9" * IP) as K_9_W,
        ("BB/9" * IP) as BB_9_W,
        ("K/BB" * IP) as K_BB_W,
        (FIP * IP) as FIP_W,
        (xFIP * IP) as xFIP_W,
        (SIERA * IP) as SIERA_W,
        (xERA * IP) as xERA_W,
        ("Barrel%" * Events) as Barrel_W,
        ("GB/FB" * Events) as GB_FB_W,
        ("GB%" * Events) as GB_pct_W,
        ("FB%" * Events) as FB_pct_W
    FROM SPGameLogFact
)


select GBS.*,
    hp.HomeSP_IP_L4A,
    hp.HomeSP_ERA_L4A,
    hp.HomeSP_SO9_L4A,
    hp.HomeSP_BB9_L4A,
    hp.HomeSP_K_BB_L4A,
    hp.HomeSP_FIP_L4A,
    hp.HomeSP_xFIP_L4A,
    hp.HomeSP_SIERA_L4A,
    hp.HomeSP_xERA_L4A,
    hp.HomeSP_Barrel_pct_L4A,
    hp.HomeSP_GB_FB_L4A,
    hp.HomeSP_GB_pct_L4A,
    hp.HomeSP_FB_pct_L4A,

    ap.AwaySP_IP_L4A,
    ap.AwaySP_ERA_L4A,
    ap.AwaySP_SO9_L4A,
    ap.AwaySP_BB9_L4A,
    ap.AwaySP_K_BB_L4A,
    ap.AwaySP_FIP_L4A,
    ap.AwaySP_xFIP_L4A,
    ap.AwaySP_SIERA_L4A,
    ap.AwaySP_xERA_L4A,
    ap.AwaySP_Barrel_pct_L4A,
    ap.AwaySP_GB_FB_L4A,
    ap.AwaySP_GB_pct_L4A,
    ap.AwaySP_FB_pct_L4A

FROM GameBoxScores_w_bullpen GBS

LEFT JOIN LATERAL(
    select 
    SUM(IP) as HomeSP_IP_L4A,
    SUM(ERA_W) / NULLIF(SUM(IP), 0) as HomeSP_ERA_L4A,
    SUM(K_9_W) / NULLIF(SUM(IP), 0) as HomeSP_SO9_L4A,
    SUM(BB_9_W) / NULLIF(SUM(IP), 0) as HomeSP_BB9_L4A,
    SUM(K_BB_W) / NULLIF(SUM(IP), 0) as HomeSP_K_BB_L4A,
    SUM(FIP_W) / NULLIF(SUM(IP), 0) as HomeSP_FIP_L4A,
    SUM(xFIP_W) / NULLIF(SUM(IP), 0) as HomeSP_xFIP_L4A,
    SUM(SIERA_W) / NULLIF(SUM(IP), 0) as HomeSP_SIERA_L4A,
    SUM(xERA_W) / NULLIF(SUM(IP), 0) as HomeSP_xERA_L4A,
    SUM(Barrel_W) / NULLIF(SUM(Events), 0) as HomeSP_Barrel_pct_L4A,
    SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeSP_GB_FB_L4A,
    SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as HomeSP_GB_pct_L4A,
    SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as HomeSP_FB_pct_L4A
    FROM(
        SELECT
        IP,
        Events,
        ERA_W,
        K_9_W,
        BB_9_W,
        K_BB_W,
        FIP_W,
        xFIP_W,
        SIERA_W,
        xERA_W,
        Barrel_W,
        GB_FB_W,
        GB_pct_W,
        FB_pct_W
        from stats_weighted sw
        where sw.playerid = GBS.HomePitcherIDFG
        and sw.season = GBS.Year
        and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC
        limit 4
    )
) hp on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(IP) as AwaySP_IP_L4A,
    SUM(ERA_W) / NULLIF(SUM(IP), 0) as AwaySP_ERA_L4A,
    SUM(K_9_W) / NULLIF(SUM(IP), 0) as AwaySP_SO9_L4A,
    SUM(BB_9_W) / NULLIF(SUM(IP), 0) as AwaySP_BB9_L4A,
    SUM(K_BB_W) / NULLIF(SUM(IP), 0) as AwaySP_K_BB_L4A,
    SUM(FIP_W) / NULLIF(SUM(IP), 0) as AwaySP_FIP_L4A,
    SUM(xFIP_W) / NULLIF(SUM(IP), 0) as AwaySP_xFIP_L4A,
    SUM(SIERA_W) / NULLIF(SUM(IP), 0) as AwaySP_SIERA_L4A,
    SUM(xERA_W) / NULLIF(SUM(IP), 0) as AwaySP_xERA_L4A,
    SUM(Barrel_W) / NULLIF(SUM(Events), 0) as AwaySP_Barrel_pct_L4A,
    SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwaySP_GB_FB_L4A,
    SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as AwaySP_GB_pct_L4A,
    SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as AwaySP_FB_pct_L4A
    FROM(
        SELECT
        IP,
        Events,
        ERA_W,
        K_9_W,
        BB_9_W,
        K_BB_W,
        FIP_W,
        xFIP_W,
        SIERA_W,
        xERA_W,
        Barrel_W,
        GB_FB_W,
        GB_pct_W,
        FB_pct_W
        from stats_weighted sw
        where sw.playerid = GBS.AwayPitcherIDFG
        and sw.season = GBS.Year
        and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC
        limit 4
    )
) ap on TRUE

""").df()

In [66]:
GameBoxScoresDAP.to_csv("GameBoxScoresDAP.csv", index=False)

# Hitting

---------------------
---------------------
---------------------

In [67]:
# Fangraphs hitting fact comes from "projectR.ipynb". Contains yearly hitter data.

fangraphs_hitting_fact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/fg_hitting_fact.csv')

In [109]:
# Adding yearly hitter data over previous two-year window for each lineup spot using lateral joins.

GameBoxScores_fghitting = duckdb.query("""
WITH stats_weighted AS (
    SELECT
        xMLBAMID,Season,AB,PA,Events,WAR,BaseRunning,(BB_pct * PA) as BB_pct_W,(K_pct * PA) as K_pct_W,(BB_K * PA) as BB_K_W,(OPS * PA) as OPS_W,(GB_FB * Events) as GB_FB_W,(GB_pct * Events) as GB_pct_W,
        (FB_pct * Events) as FB_pct_W,(wOBA * PA) as wOBA_W,(wRC_plus * PA) as wRC_plus_W,(xwOBA * PA) as xwOBA_W,(xAVG * AB) as xAVG_W,(xSLG * AB) as xSLG_W,(LA * Events) as LA_W,(Barrel_pct * Events) as Barrel_pct_W,
        maxEV,(EV90 * Events) as EV90_W
    FROM fangraphs_hitting_fact
)

select GBS.*,

    hb1.HomeB1_PA_L2,hb1.HomeB1_WAR_L2,hb1.HomeB1_BSR_L2,hb1.HomeB1_BB_pct_L2,hb1.HomeB1_K_pct_L2,hb1.HomeB1_BB_K_L2,hb1.HomeB1_OPS_L2,hb1.HomeB1_GB_FB_L2,hb1.HomeB1_GB_pct_L2,hb1.HomeB1_FB_pct_L2,
    hb1.HomeB1_wOBA_L2,hb1.HomeB1_wRC_plus_L2,hb1.HomeB1_xwOBA_L2,hb1.HomeB1_xAVG_L2,hb1.HomeB1_xSLG_L2,hb1.HomeB1_LA_L2,hb1.HomeB1_Barrel_pct_L2,hb1.HomeB1_maxEV_L2,hb1.HomeB1_EV90_L2,
    hb2.HomeB2_PA_L2,hb2.HomeB2_WAR_L2,hb2.HomeB2_BSR_L2,hb2.HomeB2_BB_pct_L2,hb2.HomeB2_K_pct_L2,hb2.HomeB2_BB_K_L2,hb2.HomeB2_OPS_L2,hb2.HomeB2_GB_FB_L2,hb2.HomeB2_GB_pct_L2,hb2.HomeB2_FB_pct_L2,
    hb2.HomeB2_wOBA_L2,hb2.HomeB2_wRC_plus_L2,hb2.HomeB2_xwOBA_L2,hb2.HomeB2_xAVG_L2,hb2.HomeB2_xSLG_L2,hb2.HomeB2_LA_L2,hb2.HomeB2_Barrel_pct_L2,hb2.HomeB2_maxEV_L2,hb2.HomeB2_EV90_L2,
    hb3.HomeB3_PA_L2,hb3.HomeB3_WAR_L2,hb3.HomeB3_BSR_L2,hb3.HomeB3_BB_pct_L2,hb3.HomeB3_K_pct_L2,hb3.HomeB3_BB_K_L2,hb3.HomeB3_OPS_L2,hb3.HomeB3_GB_FB_L2,hb3.HomeB3_GB_pct_L2,hb3.HomeB3_FB_pct_L2,
    hb3.HomeB3_wOBA_L2,hb3.HomeB3_wRC_plus_L2,hb3.HomeB3_xwOBA_L2,hb3.HomeB3_xAVG_L2,hb3.HomeB3_xSLG_L2,hb3.HomeB3_LA_L2,hb3.HomeB3_Barrel_pct_L2,hb3.HomeB3_maxEV_L2,hb3.HomeB3_EV90_L2,
    hb4.HomeB4_PA_L2,hb4.HomeB4_WAR_L2,hb4.HomeB4_BSR_L2,hb4.HomeB4_BB_pct_L2,hb4.HomeB4_K_pct_L2,hb4.HomeB4_BB_K_L2,hb4.HomeB4_OPS_L2,hb4.HomeB4_GB_FB_L2,hb4.HomeB4_GB_pct_L2,hb4.HomeB4_FB_pct_L2,
    hb4.HomeB4_wOBA_L2,hb4.HomeB4_wRC_plus_L2,hb4.HomeB4_xwOBA_L2,hb4.HomeB4_xAVG_L2,hb4.HomeB4_xSLG_L2,hb4.HomeB4_LA_L2,hb4.HomeB4_Barrel_pct_L2,hb4.HomeB4_maxEV_L2,hb4.HomeB4_EV90_L2,
    hb5.HomeB5_PA_L2,hb5.HomeB5_WAR_L2,hb5.HomeB5_BSR_L2,hb5.HomeB5_BB_pct_L2,hb5.HomeB5_K_pct_L2,hb5.HomeB5_BB_K_L2,hb5.HomeB5_OPS_L2,hb5.HomeB5_GB_FB_L2,hb5.HomeB5_GB_pct_L2,hb5.HomeB5_FB_pct_L2,
    hb5.HomeB5_wOBA_L2,hb5.HomeB5_wRC_plus_L2,hb5.HomeB5_xwOBA_L2,hb5.HomeB5_xAVG_L2,hb5.HomeB5_xSLG_L2,hb5.HomeB5_LA_L2,hb5.HomeB5_Barrel_pct_L2,hb5.HomeB5_maxEV_L2,hb5.HomeB5_EV90_L2,
    hb6.HomeB6_PA_L2,hb6.HomeB6_WAR_L2,hb6.HomeB6_BSR_L2,hb6.HomeB6_BB_pct_L2,hb6.HomeB6_K_pct_L2,hb6.HomeB6_BB_K_L2,hb6.HomeB6_OPS_L2,hb6.HomeB6_GB_FB_L2,hb6.HomeB6_GB_pct_L2,hb6.HomeB6_FB_pct_L2,
    hb6.HomeB6_wOBA_L2,hb6.HomeB6_wRC_plus_L2,hb6.HomeB6_xwOBA_L2,hb6.HomeB6_xAVG_L2,hb6.HomeB6_xSLG_L2,hb6.HomeB6_LA_L2,hb6.HomeB6_Barrel_pct_L2,hb6.HomeB6_maxEV_L2,hb6.HomeB6_EV90_L2,
    hb7.HomeB7_PA_L2,hb7.HomeB7_WAR_L2,hb7.HomeB7_BSR_L2,hb7.HomeB7_BB_pct_L2,hb7.HomeB7_K_pct_L2,hb7.HomeB7_BB_K_L2,hb7.HomeB7_OPS_L2,hb7.HomeB7_GB_FB_L2,hb7.HomeB7_GB_pct_L2,hb7.HomeB7_FB_pct_L2,
    hb7.HomeB7_wOBA_L2,hb7.HomeB7_wRC_plus_L2,hb7.HomeB7_xwOBA_L2,hb7.HomeB7_xAVG_L2,hb7.HomeB7_xSLG_L2,hb7.HomeB7_LA_L2,hb7.HomeB7_Barrel_pct_L2,hb7.HomeB7_maxEV_L2,hb7.HomeB7_EV90_L2,
    hb8.HomeB8_PA_L2,hb8.HomeB8_WAR_L2,hb8.HomeB8_BSR_L2,hb8.HomeB8_BB_pct_L2,hb8.HomeB8_K_pct_L2,hb8.HomeB8_BB_K_L2,hb8.HomeB8_OPS_L2,hb8.HomeB8_GB_FB_L2,hb8.HomeB8_GB_pct_L2,hb8.HomeB8_FB_pct_L2,
    hb8.HomeB8_wOBA_L2,hb8.HomeB8_wRC_plus_L2,hb8.HomeB8_xwOBA_L2,hb8.HomeB8_xAVG_L2,hb8.HomeB8_xSLG_L2,hb8.HomeB8_LA_L2,hb8.HomeB8_Barrel_pct_L2,hb8.HomeB8_maxEV_L2,hb8.HomeB8_EV90_L2,
    hb9.HomeB9_PA_L2,hb9.HomeB9_WAR_L2,hb9.HomeB9_BSR_L2,hb9.HomeB9_BB_pct_L2,hb9.HomeB9_K_pct_L2,hb9.HomeB9_BB_K_L2,hb9.HomeB9_OPS_L2,hb9.HomeB9_GB_FB_L2,hb9.HomeB9_GB_pct_L2,hb9.HomeB9_FB_pct_L2,
    hb9.HomeB9_wOBA_L2,hb9.HomeB9_wRC_plus_L2,hb9.HomeB9_xwOBA_L2,hb9.HomeB9_xAVG_L2,hb9.HomeB9_xSLG_L2,hb9.HomeB9_LA_L2,hb9.HomeB9_Barrel_pct_L2,hb9.HomeB9_maxEV_L2,hb9.HomeB9_EV90_L2,
    ab1.AwayB1_PA_L2,ab1.AwayB1_WAR_L2,ab1.AwayB1_BSR_L2,ab1.AwayB1_BB_pct_L2,ab1.AwayB1_K_pct_L2,ab1.AwayB1_BB_K_L2,ab1.AwayB1_OPS_L2,ab1.AwayB1_GB_FB_L2,ab1.AwayB1_GB_pct_L2,ab1.AwayB1_FB_pct_L2,
    ab1.AwayB1_wOBA_L2,ab1.AwayB1_wRC_plus_L2,ab1.AwayB1_xwOBA_L2,ab1.AwayB1_xAVG_L2,ab1.AwayB1_xSLG_L2,ab1.AwayB1_LA_L2,ab1.AwayB1_Barrel_pct_L2,ab1.AwayB1_maxEV_L2,ab1.AwayB1_EV90_L2,
    ab2.AwayB2_PA_L2,ab2.AwayB2_WAR_L2,ab2.AwayB2_BSR_L2,ab2.AwayB2_BB_pct_L2,ab2.AwayB2_K_pct_L2,ab2.AwayB2_BB_K_L2,ab2.AwayB2_OPS_L2,ab2.AwayB2_GB_FB_L2,ab2.AwayB2_GB_pct_L2,ab2.AwayB2_FB_pct_L2,
    ab2.AwayB2_wOBA_L2,ab2.AwayB2_wRC_plus_L2,ab2.AwayB2_xwOBA_L2,ab2.AwayB2_xAVG_L2,ab2.AwayB2_xSLG_L2,ab2.AwayB2_LA_L2,ab2.AwayB2_Barrel_pct_L2,ab2.AwayB2_maxEV_L2,ab2.AwayB2_EV90_L2,
    ab3.AwayB3_PA_L2,ab3.AwayB3_WAR_L2,ab3.AwayB3_BSR_L2,ab3.AwayB3_BB_pct_L2,ab3.AwayB3_K_pct_L2,ab3.AwayB3_BB_K_L2,ab3.AwayB3_OPS_L2,ab3.AwayB3_GB_FB_L2,ab3.AwayB3_GB_pct_L2,ab3.AwayB3_FB_pct_L2,
    ab3.AwayB3_wOBA_L2,ab3.AwayB3_wRC_plus_L2,ab3.AwayB3_xwOBA_L2,ab3.AwayB3_xAVG_L2,ab3.AwayB3_xSLG_L2,ab3.AwayB3_LA_L2,ab3.AwayB3_Barrel_pct_L2,ab3.AwayB3_maxEV_L2,ab3.AwayB3_EV90_L2,
    ab4.AwayB4_PA_L2,ab4.AwayB4_WAR_L2,ab4.AwayB4_BSR_L2,ab4.AwayB4_BB_pct_L2,ab4.AwayB4_K_pct_L2,ab4.AwayB4_BB_K_L2,ab4.AwayB4_OPS_L2,ab4.AwayB4_GB_FB_L2,ab4.AwayB4_GB_pct_L2,ab4.AwayB4_FB_pct_L2,
    ab4.AwayB4_wOBA_L2,ab4.AwayB4_wRC_plus_L2,ab4.AwayB4_xwOBA_L2,ab4.AwayB4_xAVG_L2,ab4.AwayB4_xSLG_L2,ab4.AwayB4_LA_L2,ab4.AwayB4_Barrel_pct_L2,ab4.AwayB4_maxEV_L2,ab4.AwayB4_EV90_L2,
    ab5.AwayB5_PA_L2,ab5.AwayB5_WAR_L2,ab5.AwayB5_BSR_L2,ab5.AwayB5_BB_pct_L2,ab5.AwayB5_K_pct_L2,ab5.AwayB5_BB_K_L2,ab5.AwayB5_OPS_L2,ab5.AwayB5_GB_FB_L2,ab5.AwayB5_GB_pct_L2,ab5.AwayB5_FB_pct_L2,
    ab5.AwayB5_wOBA_L2,ab5.AwayB5_wRC_plus_L2,ab5.AwayB5_xwOBA_L2,ab5.AwayB5_xAVG_L2,ab5.AwayB5_xSLG_L2,ab5.AwayB5_LA_L2,ab5.AwayB5_Barrel_pct_L2,ab5.AwayB5_maxEV_L2,ab5.AwayB5_EV90_L2,
    ab6.AwayB6_PA_L2,ab6.AwayB6_WAR_L2,ab6.AwayB6_BSR_L2,ab6.AwayB6_BB_pct_L2,ab6.AwayB6_K_pct_L2,ab6.AwayB6_BB_K_L2,ab6.AwayB6_OPS_L2,ab6.AwayB6_GB_FB_L2,ab6.AwayB6_GB_pct_L2,ab6.AwayB6_FB_pct_L2,
    ab6.AwayB6_wOBA_L2,ab6.AwayB6_wRC_plus_L2,ab6.AwayB6_xwOBA_L2,ab6.AwayB6_xAVG_L2,ab6.AwayB6_xSLG_L2,ab6.AwayB6_LA_L2,ab6.AwayB6_Barrel_pct_L2,ab6.AwayB6_maxEV_L2,ab6.AwayB6_EV90_L2,
    ab7.AwayB7_PA_L2,ab7.AwayB7_WAR_L2,ab7.AwayB7_BSR_L2,ab7.AwayB7_BB_pct_L2,ab7.AwayB7_K_pct_L2,ab7.AwayB7_BB_K_L2,ab7.AwayB7_OPS_L2,ab7.AwayB7_GB_FB_L2,ab7.AwayB7_GB_pct_L2,ab7.AwayB7_FB_pct_L2,
    ab7.AwayB7_wOBA_L2,ab7.AwayB7_wRC_plus_L2,ab7.AwayB7_xwOBA_L2,ab7.AwayB7_xAVG_L2,ab7.AwayB7_xSLG_L2,ab7.AwayB7_LA_L2,ab7.AwayB7_Barrel_pct_L2,ab7.AwayB7_maxEV_L2,ab7.AwayB7_EV90_L2,
    ab8.AwayB8_PA_L2,ab8.AwayB8_WAR_L2,ab8.AwayB8_BSR_L2,ab8.AwayB8_BB_pct_L2,ab8.AwayB8_K_pct_L2,ab8.AwayB8_BB_K_L2,ab8.AwayB8_OPS_L2,ab8.AwayB8_GB_FB_L2,ab8.AwayB8_GB_pct_L2,ab8.AwayB8_FB_pct_L2,
    ab8.AwayB8_wOBA_L2,ab8.AwayB8_wRC_plus_L2,ab8.AwayB8_xwOBA_L2,ab8.AwayB8_xAVG_L2,ab8.AwayB8_xSLG_L2,ab8.AwayB8_LA_L2,ab8.AwayB8_Barrel_pct_L2,ab8.AwayB8_maxEV_L2,ab8.AwayB8_EV90_L2,
    ab9.AwayB9_PA_L2,ab9.AwayB9_WAR_L2,ab9.AwayB9_BSR_L2,ab9.AwayB9_BB_pct_L2,ab9.AwayB9_K_pct_L2,ab9.AwayB9_BB_K_L2,ab9.AwayB9_OPS_L2,ab9.AwayB9_GB_FB_L2,ab9.AwayB9_GB_pct_L2,ab9.AwayB9_FB_pct_L2,
    ab9.AwayB9_wOBA_L2,ab9.AwayB9_wRC_plus_L2,ab9.AwayB9_xwOBA_L2,ab9.AwayB9_xAVG_L2,ab9.AwayB9_xSLG_L2,ab9.AwayB9_LA_L2,ab9.AwayB9_Barrel_pct_L2,ab9.AwayB9_maxEV_L2,ab9.AwayB9_EV90_L2

FROM GameBoxScoresDAP GBS

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB1_PA_L2,SUM(WAR) as HomeB1_WAR_L2,SUM(BaseRunning) as HomeB1_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB1_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB1_K_pct_L2,
    SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB1_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB1_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB1_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as HomeB1_GB_pct_L2,
    SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as HomeB1_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB1_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB1_wRC_plus_L2,
    SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB1_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB1_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB1_xSLG_L2,
    SUM(LA_W) / NULLIF(SUM(Events), 0) as HomeB1_LA_L2, SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB1_Barrel_pct_L2,MAX(maxEV) as HomeB1_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as HomeB1_EV90_L2
    FROM(
        SELECT
        PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
        from stats_weighted sw
        where sw.xMLBAMID = GBS.HomeBatter1IDMLB and sw.Season < GBS.Year
        order by sw.Season DESC
        limit 2
    )
) hb1 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB2_PA_L2,SUM(WAR) as HomeB2_WAR_L2,SUM(BaseRunning) as HomeB2_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB2_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB2_K_pct_L2,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB2_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB2_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB2_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as HomeB2_GB_pct_L2,
   SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as HomeB2_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB2_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB2_wRC_plus_L2,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB2_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB2_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB2_xSLG_L2,
   SUM(LA_W) / NULLIF(SUM(Events), 0) as HomeB2_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB2_Barrel_pct_L2,MAX(maxEV) as HomeB2_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as HomeB2_EV90_L2
   FROM(
       SELECT
       PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
       from stats_weighted sw
       where sw.xMLBAMID = GBS.HomeBatter2IDMLB and sw.Season < GBS.Year
       order by sw.Season DESC
       limit 2
   )
) hb2 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB3_PA_L2,SUM(WAR) as HomeB3_WAR_L2,SUM(BaseRunning) as HomeB3_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB3_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB3_K_pct_L2,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB3_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB3_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB3_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as HomeB3_GB_pct_L2,
   SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as HomeB3_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB3_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB3_wRC_plus_L2,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB3_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB3_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB3_xSLG_L2,
   SUM(LA_W) / NULLIF(SUM(Events), 0) as HomeB3_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB3_Barrel_pct_L2,MAX(maxEV) as HomeB3_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as HomeB3_EV90_L2
   FROM(
       SELECT
       PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
       from stats_weighted sw
       where sw.xMLBAMID = GBS.HomeBatter3IDMLB and sw.Season < GBS.Year
       order by sw.Season DESC
       limit 2
   )
) hb3 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB4_PA_L2,SUM(WAR) as HomeB4_WAR_L2,SUM(BaseRunning) as HomeB4_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB4_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB4_K_pct_L2,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB4_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB4_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB4_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as HomeB4_GB_pct_L2,
   SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as HomeB4_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB4_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB4_wRC_plus_L2,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB4_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB4_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB4_xSLG_L2,
   SUM(LA_W) / NULLIF(SUM(Events), 0) as HomeB4_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB4_Barrel_pct_L2,MAX(maxEV) as HomeB4_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as HomeB4_EV90_L2
   FROM(
       SELECT
       PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
       from stats_weighted sw
       where sw.xMLBAMID = GBS.HomeBatter4IDMLB and sw.Season < GBS.Year
       order by sw.Season DESC
       limit 2
   )
) hb4 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB5_PA_L2,SUM(WAR) as HomeB5_WAR_L2,SUM(BaseRunning) as HomeB5_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB5_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB5_K_pct_L2,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB5_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB5_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB5_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as HomeB5_GB_pct_L2,
   SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as HomeB5_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB5_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB5_wRC_plus_L2,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB5_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB5_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB5_xSLG_L2,
   SUM(LA_W) / NULLIF(SUM(Events), 0) as HomeB5_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB5_Barrel_pct_L2,MAX(maxEV) as HomeB5_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as HomeB5_EV90_L2
   FROM(
       SELECT
       PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
       from stats_weighted sw
       where sw.xMLBAMID = GBS.HomeBatter5IDMLB and sw.Season < GBS.Year
       order by sw.Season DESC
       limit 2
   )
) hb5 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB6_PA_L2,SUM(WAR) as HomeB6_WAR_L2,SUM(BaseRunning) as HomeB6_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB6_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB6_K_pct_L2,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB6_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB6_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB6_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as HomeB6_GB_pct_L2,
   SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as HomeB6_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB6_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB6_wRC_plus_L2,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB6_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB6_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB6_xSLG_L2,
   SUM(LA_W) / NULLIF(SUM(Events), 0) as HomeB6_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB6_Barrel_pct_L2,MAX(maxEV) as HomeB6_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as HomeB6_EV90_L2
   FROM(
       SELECT
       PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
       from stats_weighted sw
       where sw.xMLBAMID = GBS.HomeBatter6IDMLB and sw.Season < GBS.Year
       order by sw.Season DESC
       limit 2
   )
) hb6 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB7_PA_L2,SUM(WAR) as HomeB7_WAR_L2,SUM(BaseRunning) as HomeB7_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB7_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB7_K_pct_L2,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB7_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB7_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB7_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as HomeB7_GB_pct_L2,
   SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as HomeB7_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB7_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB7_wRC_plus_L2,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB7_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB7_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB7_xSLG_L2,
   SUM(LA_W) / NULLIF(SUM(Events), 0) as HomeB7_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB7_Barrel_pct_L2,MAX(maxEV) as HomeB7_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as HomeB7_EV90_L2
   FROM(
       SELECT
       PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
       from stats_weighted sw
       where sw.xMLBAMID = GBS.HomeBatter7IDMLB and sw.Season < GBS.Year
       order by sw.Season DESC
       limit 2
   )
) hb7 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB8_PA_L2,SUM(WAR) as HomeB8_WAR_L2,SUM(BaseRunning) as HomeB8_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB8_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB8_K_pct_L2,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB8_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB8_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB8_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as HomeB8_GB_pct_L2,
   SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as HomeB8_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB8_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB8_wRC_plus_L2,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB8_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB8_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB8_xSLG_L2,
   SUM(LA_W) / NULLIF(SUM(Events), 0) as HomeB8_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB8_Barrel_pct_L2,MAX(maxEV) as HomeB8_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as HomeB8_EV90_L2
   FROM(
       SELECT
       PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
       from stats_weighted sw
       where sw.xMLBAMID = GBS.HomeBatter8IDMLB and sw.Season < GBS.Year
       order by sw.Season DESC
       limit 2
   )
) hb8 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB9_PA_L2,SUM(WAR) as HomeB9_WAR_L2,SUM(BaseRunning) as HomeB9_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB9_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB9_K_pct_L2,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB9_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB9_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB9_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as HomeB9_GB_pct_L2,
   SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as HomeB9_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB9_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB9_wRC_plus_L2,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB9_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB9_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB9_xSLG_L2,
   SUM(LA_W) / NULLIF(SUM(Events), 0) as HomeB9_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB9_Barrel_pct_L2,MAX(maxEV) as HomeB9_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as HomeB9_EV90_L2
   FROM(
       SELECT
       PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
       from stats_weighted sw
       where sw.xMLBAMID = GBS.HomeBatter9IDMLB and sw.Season < GBS.Year
       order by sw.Season DESC
       limit 2
   )
) hb9 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as AwayB1_PA_L2,SUM(WAR) as AwayB1_WAR_L2,SUM(BaseRunning) as AwayB1_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB1_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB1_K_pct_L2,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB1_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB1_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB1_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as AwayB1_GB_pct_L2,
   SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as AwayB1_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB1_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB1_wRC_plus_L2,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB1_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB1_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB1_xSLG_L2,
   SUM(LA_W) / NULLIF(SUM(Events), 0) as AwayB1_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB1_Barrel_pct_L2,MAX(maxEV) as AwayB1_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as AwayB1_EV90_L2
   FROM(
       SELECT
       PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
       from stats_weighted sw
       where sw.xMLBAMID = GBS.AwayBatter1IDMLB and sw.Season < GBS.Year
       order by sw.Season DESC
       limit 2
   )
) ab1 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB2_PA_L2,SUM(WAR) as AwayB2_WAR_L2,SUM(BaseRunning) as AwayB2_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB2_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB2_K_pct_L2,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB2_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB2_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB2_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as AwayB2_GB_pct_L2,
  SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as AwayB2_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB2_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB2_wRC_plus_L2,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB2_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB2_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB2_xSLG_L2,
  SUM(LA_W) / NULLIF(SUM(Events), 0) as AwayB2_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB2_Barrel_pct_L2,MAX(maxEV) as AwayB2_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as AwayB2_EV90_L2
  FROM(
      SELECT
      PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
      from stats_weighted sw
      where sw.xMLBAMID = GBS.AwayBatter2IDMLB and sw.Season < GBS.Year
      order by sw.Season DESC
      limit 2
  )
) ab2 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB3_PA_L2,SUM(WAR) as AwayB3_WAR_L2,SUM(BaseRunning) as AwayB3_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB3_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB3_K_pct_L2,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB3_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB3_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB3_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as AwayB3_GB_pct_L2,
  SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as AwayB3_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB3_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB3_wRC_plus_L2,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB3_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB3_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB3_xSLG_L2,
  SUM(LA_W) / NULLIF(SUM(Events), 0) as AwayB3_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB3_Barrel_pct_L2,MAX(maxEV) as AwayB3_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as AwayB3_EV90_L2
  FROM(
      SELECT
      PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
      from stats_weighted sw
      where sw.xMLBAMID = GBS.AwayBatter3IDMLB and sw.Season < GBS.Year
      order by sw.Season DESC
      limit 2
  )
) ab3 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB4_PA_L2,SUM(WAR) as AwayB4_WAR_L2,SUM(BaseRunning) as AwayB4_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB4_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB4_K_pct_L2,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB4_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB4_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB4_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as AwayB4_GB_pct_L2,
  SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as AwayB4_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB4_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB4_wRC_plus_L2,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB4_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB4_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB4_xSLG_L2,
  SUM(LA_W) / NULLIF(SUM(Events), 0) as AwayB4_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB4_Barrel_pct_L2,MAX(maxEV) as AwayB4_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as AwayB4_EV90_L2
  FROM(
      SELECT
      PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
      from stats_weighted sw
      where sw.xMLBAMID = GBS.AwayBatter4IDMLB and sw.Season < GBS.Year
      order by sw.Season DESC
      limit 2
  )
) ab4 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB5_PA_L2,SUM(WAR) as AwayB5_WAR_L2,SUM(BaseRunning) as AwayB5_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB5_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB5_K_pct_L2,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB5_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB5_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB5_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as AwayB5_GB_pct_L2,
  SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as AwayB5_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB5_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB5_wRC_plus_L2,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB5_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB5_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB5_xSLG_L2,
  SUM(LA_W) / NULLIF(SUM(Events), 0) as AwayB5_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB5_Barrel_pct_L2,MAX(maxEV) as AwayB5_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as AwayB5_EV90_L2
  FROM(
      SELECT
      PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
      from stats_weighted sw
      where sw.xMLBAMID = GBS.AwayBatter5IDMLB and sw.Season < GBS.Year
      order by sw.Season DESC
      limit 2
  )
) ab5 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB6_PA_L2,SUM(WAR) as AwayB6_WAR_L2,SUM(BaseRunning) as AwayB6_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB6_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB6_K_pct_L2,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB6_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB6_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB6_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as AwayB6_GB_pct_L2,
  SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as AwayB6_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB6_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB6_wRC_plus_L2,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB6_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB6_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB6_xSLG_L2,
  SUM(LA_W) / NULLIF(SUM(Events), 0) as AwayB6_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB6_Barrel_pct_L2,MAX(maxEV) as AwayB6_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as AwayB6_EV90_L2
  FROM(
      SELECT
      PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
      from stats_weighted sw
      where sw.xMLBAMID = GBS.AwayBatter6IDMLB and sw.Season < GBS.Year
      order by sw.Season DESC
      limit 2
  )
) ab6 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB7_PA_L2,SUM(WAR) as AwayB7_WAR_L2,SUM(BaseRunning) as AwayB7_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB7_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB7_K_pct_L2,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB7_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB7_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB7_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as AwayB7_GB_pct_L2,
  SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as AwayB7_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB7_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB7_wRC_plus_L2,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB7_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB7_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB7_xSLG_L2,
  SUM(LA_W) / NULLIF(SUM(Events), 0) as AwayB7_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB7_Barrel_pct_L2,MAX(maxEV) as AwayB7_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as AwayB7_EV90_L2
  FROM(
      SELECT
      PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
      from stats_weighted sw
      where sw.xMLBAMID = GBS.AwayBatter7IDMLB and sw.Season < GBS.Year
      order by sw.Season DESC
      limit 2
  )
) ab7 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB8_PA_L2,SUM(WAR) as AwayB8_WAR_L2,SUM(BaseRunning) as AwayB8_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB8_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB8_K_pct_L2,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB8_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB8_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB8_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as AwayB8_GB_pct_L2,
  SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as AwayB8_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB8_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB8_wRC_plus_L2,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB8_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB8_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB8_xSLG_L2,
  SUM(LA_W) / NULLIF(SUM(Events), 0) as AwayB8_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB8_Barrel_pct_L2,MAX(maxEV) as AwayB8_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as AwayB8_EV90_L2
  FROM(
      SELECT
      PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
      from stats_weighted sw
      where sw.xMLBAMID = GBS.AwayBatter8IDMLB and sw.Season < GBS.Year
      order by sw.Season DESC
      limit 2
  )
) ab8 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB9_PA_L2,SUM(WAR) as AwayB9_WAR_L2,SUM(BaseRunning) as AwayB9_BSR_L2,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB9_BB_pct_L2,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB9_K_pct_L2,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB9_BB_K_L2,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB9_OPS_L2,SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB9_GB_FB_L2,SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as AwayB9_GB_pct_L2,
  SUM(FB_pct_W) / NULLIF(SUM(Events), 0) as AwayB9_FB_pct_L2,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB9_wOBA_L2,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB9_wRC_plus_L2,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB9_xwOBA_L2,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB9_xAVG_L2,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB9_xSLG_L2,
  SUM(LA_W) / NULLIF(SUM(Events), 0) as AwayB9_LA_L2,SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB9_Barrel_pct_L2,MAX(maxEV) as AwayB9_maxEV_L2,SUM(EV90_W) / NULLIF(SUM(Events), 0) as AwayB9_EV90_L2
  FROM(
      SELECT
      PA,AB,Events,WAR,BaseRunning,BB_pct_W,K_pct_W,BB_K_W,OPS_W,GB_FB_W,GB_pct_W,FB_pct_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,LA_W,Barrel_pct_W,maxEV,EV90_W
      from stats_weighted sw
      where sw.xMLBAMID = GBS.AwayBatter9IDMLB and sw.Season < GBS.Year
      order by sw.Season DESC
      limit 2
  )
) ab9 on TRUE

""").df()

In [69]:
GameBoxScores_fghitting.to_csv("GBS_fghitting", index = False)

In [13]:
# Loading hitting data from baseball prospectus. Ignore this process and simply load in drc_map_clean from files to view final product. Process shown below.

drc17 = pd.read_csv('/Users/owendrummond/Downloads/DRC2017.csv')
drc18 = pd.read_csv('/Users/owendrummond/Downloads/DRC2018.csv')
drc19 = pd.read_csv('/Users/owendrummond/Downloads/DRC2019.csv')
drc20 = pd.read_csv('/Users/owendrummond/Downloads/DRC2020.csv')
drc21 = pd.read_csv('/Users/owendrummond/Downloads/DRC2021.csv')
drc22 = pd.read_csv('/Users/owendrummond/Downloads/DRC2022.csv')
drc23 = pd.read_csv('/Users/owendrummond/Downloads/DRC2023.csv')
drc24 = pd.read_csv('/Users/owendrummond/Downloads/DRC2024.csv')
drc25 = pd.read_csv('/Users/owendrummond/Downloads/DRC2025.csv')

drc_map = pd.concat([drc17, drc18, drc19, drc20, drc21, drc22, drc23, drc24, drc25])
print(len(drc17) + len(drc18) + len(drc19) + len(drc20) + len(drc21) + len(drc22) + len(drc23) + len(drc24) + len(drc25))

26658


In [14]:
# Cleaning month values and adding year field for baseball prospectus data.

drc_map_clean = duckdb.query("""
        with sq as(
        select 
        bpid,
        mlbid,
        Name,
        Season as Year,
        CASE
            WHEN "Month" = 'Mar/Apr' THEN '04'
            WHEN "Month" LIKE '%May%' THEN '05'
            WHEN "Month" LIKE '%June%' THEN '06' 
            WHEN "Month" LIKE '%July%' THEN '07' 
            WHEN "Month" LIKE '%August%' THEN '08' 
            WHEN "Month" = 'Sep/Oct' THEN '09' 
        END AS Month,
        PA,
        AB,
        OBP,
        OPS,
        "DRC+" as DRC_plus,
        "K%",
        "BB%"
        FROM drc_map),
        
        add_date as(
            select *,
            date_trunc('Month', make_date(CAST(Year AS INTEGER), CAST(Month AS INTEGER), 1)) + INTERVAL 1 MONTH - INTERVAL 1 DAY AS CompDate
            from sq
        )
        
        select * from add_date
""").df()

In [16]:
drc_map_clean.to_csv('drc_map_clean.csv', index=False)

In [110]:
# Adding baseball prospectus L5M window hitting data to GameBoxScores for each lineup spot.

GameBoxScores_bphitting = duckdb.query("""
with stats_weighted as (
        SELECT bpid, mlbid, Year, Month, PA, CompDate, (OBP * PA) as OBP_W, (OPS * PA) as OPS_W, (DRC_plus * PA) as DRC_plus_W,
        ("K%" * PA) as Kpct_W, ("BB%" * PA) as BBpct_W
    FROM drc_map_clean
)

select GBS.*,

hb1.HomeB1_PA_L5M,hb1.HomeB1_OBP_L5M,hb1.HomeB1_OPS_L5M,hb1.HomeB1_DRC_plus_L5M,hb1.HomeB1_Kpct_L5M,hb1.HomeB1_BBpct_L5M,
hb2.HomeB2_PA_L5M,hb2.HomeB2_OBP_L5M,hb2.HomeB2_OPS_L5M,hb2.HomeB2_DRC_plus_L5M,hb2.HomeB2_Kpct_L5M,hb2.HomeB2_BBpct_L5M,
hb3.HomeB3_PA_L5M,hb3.HomeB3_OBP_L5M,hb3.HomeB3_OPS_L5M,hb3.HomeB3_DRC_plus_L5M,hb3.HomeB3_Kpct_L5M,hb3.HomeB3_BBpct_L5M,
hb4.HomeB4_PA_L5M,hb4.HomeB4_OBP_L5M,hb4.HomeB4_OPS_L5M,hb4.HomeB4_DRC_plus_L5M,hb4.HomeB4_Kpct_L5M,hb4.HomeB4_BBpct_L5M,
hb5.HomeB5_PA_L5M,hb5.HomeB5_OBP_L5M,hb5.HomeB5_OPS_L5M,hb5.HomeB5_DRC_plus_L5M,hb5.HomeB5_Kpct_L5M,hb5.HomeB5_BBpct_L5M,
hb6.HomeB6_PA_L5M,hb6.HomeB6_OBP_L5M,hb6.HomeB6_OPS_L5M,hb6.HomeB6_DRC_plus_L5M,hb6.HomeB6_Kpct_L5M,hb6.HomeB6_BBpct_L5M,
hb7.HomeB7_PA_L5M,hb7.HomeB7_OBP_L5M,hb7.HomeB7_OPS_L5M,hb7.HomeB7_DRC_plus_L5M,hb7.HomeB7_Kpct_L5M,hb7.HomeB7_BBpct_L5M,
hb8.HomeB8_PA_L5M,hb8.HomeB8_OBP_L5M,hb8.HomeB8_OPS_L5M,hb8.HomeB8_DRC_plus_L5M,hb8.HomeB8_Kpct_L5M,hb8.HomeB8_BBpct_L5M,
hb9.HomeB9_PA_L5M,hb9.HomeB9_OBP_L5M,hb9.HomeB9_OPS_L5M,hb9.HomeB9_DRC_plus_L5M,hb9.HomeB9_Kpct_L5M,hb9.HomeB9_BBpct_L5M,

ab1.AwayB1_PA_L5M,ab1.AwayB1_OBP_L5M,ab1.AwayB1_OPS_L5M,ab1.AwayB1_DRC_plus_L5M,ab1.AwayB1_Kpct_L5M,ab1.AwayB1_BBpct_L5M,
ab2.AwayB2_PA_L5M,ab2.AwayB2_OBP_L5M,ab2.AwayB2_OPS_L5M,ab2.AwayB2_DRC_plus_L5M,ab2.AwayB2_Kpct_L5M,ab2.AwayB2_BBpct_L5M,
ab3.AwayB3_PA_L5M,ab3.AwayB3_OBP_L5M,ab3.AwayB3_OPS_L5M,ab3.AwayB3_DRC_plus_L5M,ab3.AwayB3_Kpct_L5M,ab3.AwayB3_BBpct_L5M,
ab4.AwayB4_PA_L5M,ab4.AwayB4_OBP_L5M,ab4.AwayB4_OPS_L5M,ab4.AwayB4_DRC_plus_L5M,ab4.AwayB4_Kpct_L5M,ab4.AwayB4_BBpct_L5M,
ab5.AwayB5_PA_L5M,ab5.AwayB5_OBP_L5M,ab5.AwayB5_OPS_L5M,ab5.AwayB5_DRC_plus_L5M,ab5.AwayB5_Kpct_L5M,ab5.AwayB5_BBpct_L5M,
ab6.AwayB6_PA_L5M,ab6.AwayB6_OBP_L5M,ab6.AwayB6_OPS_L5M,ab6.AwayB6_DRC_plus_L5M,ab6.AwayB6_Kpct_L5M,ab6.AwayB6_BBpct_L5M,
ab7.AwayB7_PA_L5M,ab7.AwayB7_OBP_L5M,ab7.AwayB7_OPS_L5M,ab7.AwayB7_DRC_plus_L5M,ab7.AwayB7_Kpct_L5M,ab7.AwayB7_BBpct_L5M,
ab8.AwayB8_PA_L5M,ab8.AwayB8_OBP_L5M,ab8.AwayB8_OPS_L5M,ab8.AwayB8_DRC_plus_L5M,ab8.AwayB8_Kpct_L5M,ab8.AwayB8_BBpct_L5M,
ab9.AwayB9_PA_L5M,ab9.AwayB9_OBP_L5M,ab9.AwayB9_OPS_L5M,ab9.AwayB9_DRC_plus_L5M,ab9.AwayB9_Kpct_L5M,ab9.AwayB9_BBpct_L5M

FROM GameBoxScores_fghitting GBS

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB1_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB1_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB1_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB1_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB1_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB1_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter1IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) hb1 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB2_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB2_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB2_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB2_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB2_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB2_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter2IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) hb2 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB3_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB3_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB3_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB3_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB3_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB3_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter3IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) hb3 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB4_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB4_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB4_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB4_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB4_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB4_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter4IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) hb4 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB5_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB5_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB5_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB5_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB5_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB5_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter5IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) hb5 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB6_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB6_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB6_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB6_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB6_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB6_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter6IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) hb6 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB7_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB7_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB7_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB7_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB7_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB7_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter7IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) hb7 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB8_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB8_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB8_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB8_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB8_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB8_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter8IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) hb8 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB9_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB9_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB9_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB9_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB9_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB9_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter9IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) hb9 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB1_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB1_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB1_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB1_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB1_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB1_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter1IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) ab1 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB2_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB2_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB2_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB2_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB2_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB2_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter2IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) ab2 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB3_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB3_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB3_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB3_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB3_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB3_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter3IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) ab3 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB4_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB4_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB4_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB4_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB4_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB4_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter4IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) ab4 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB5_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB5_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB5_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB5_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB5_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB5_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter5IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) ab5 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB6_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB6_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB6_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB6_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB6_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB6_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter6IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) ab6 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB7_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB7_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB7_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB7_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB7_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB7_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter7IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) ab7 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB8_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB8_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB8_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB8_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB8_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB8_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter8IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) ab8 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB9_PA_L5M, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB9_OBP_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB9_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB9_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB9_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB9_BBpct_L5M
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter9IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) ab9 on TRUE

""").df()

In [ ]:
GameBoxScores_bphitting

In [17]:
# Same process as before, but for platoon hitting. See drcp_map_clean CSV file for data. Operation shown in following cell.

drcp17 = pd.read_csv('/Users/owendrummond/Downloads/DRCP2017.csv')
drcp18 = pd.read_csv('/Users/owendrummond/Downloads/DRCP2018.csv')
drcp19 = pd.read_csv('/Users/owendrummond/Downloads/DRCP2019.csv')
drcp20 = pd.read_csv('/Users/owendrummond/Downloads/DRCP2020.csv')
drcp21 = pd.read_csv('/Users/owendrummond/Downloads/DRCP2021.csv')
drcp22 = pd.read_csv('/Users/owendrummond/Downloads/DRCP2022.csv')
drcp23 = pd.read_csv('/Users/owendrummond/Downloads/DRCP2023.csv')
drcp24 = pd.read_csv('/Users/owendrummond/Downloads/DRCP2024.csv')
drcp25 = pd.read_csv('/Users/owendrummond/Downloads/DRCP2025.csv')

drcp_map = pd.concat([drcp17, drcp18, drcp19, drcp20, drcp21, drcp22, drcp23, drcp24, drcp25])
print(len(drcp17) + len(drcp18) + len(drcp19) + len(drcp20) + len(drcp21) + len(drcp22) + len(drcp23) + len(drcp24) + len(drcp25))

46200


In [18]:
# Cleaning month field and adding year field for baseball prospectus hitting stats.

drcp_map_clean = duckdb.query("""
        with sq as(
        select 
        bpid,
        mlbid,
        Name,
        Season as Year,
        CASE
            WHEN "Month" = 'Mar/Apr' THEN '04'
            WHEN "Month" LIKE '%May%' THEN '05'
            WHEN "Month" LIKE '%June%' THEN '06' 
            WHEN "Month" LIKE '%July%' THEN '07' 
            WHEN "Month" LIKE '%August%' THEN '08' 
            WHEN "Month" = 'Sep/Oct' THEN '09' 
        END AS Month,
        CASE
            WHEN Platoon = 'vs_RHP' THEN 'R'
            WHEN Platoon = 'vs_LHP' THEN 'L'
        END AS PlatoonHand,
        PA,
        AB,
        OBP,
        OPS,
        "DRC+" as DRC_plus,
        "K%",
        "BB%"
        FROM drcp_map),
        
        add_date as(
            select *,
            date_trunc('Month', make_date(CAST(Year AS INTEGER), CAST(Month AS INTEGER), 1)) + INTERVAL 1 MONTH - INTERVAL 1 DAY AS CompDate
            from sq
        )
        
        select * from add_date
""").df()

In [19]:
drcp_map_clean.to_csv('drcp_map_clean.csv', index=False)

In [111]:
# Adding baseball prospectus L5M platoon window hitting data to GameBoxScores for each lineup spot.

GameBoxScores_phitting = duckdb.query("""
with stats_weighted as (
        SELECT bpid, mlbid, Year, Month, PlatoonHand, PA, CompDate, (OBP * PA) as OBP_W, (OPS * PA) as OPS_W, (DRC_plus * PA) as DRC_plus_W,
        ("K%" * PA) as Kpct_W, ("BB%" * PA) as BBpct_W
    FROM drcp_map_clean
)

select GBS.*,

hb1.HomeB1_PA_L5M_P,hb1.HomeB1_OBP_L5M_P,hb1.HomeB1_OPS_L5M_P,hb1.HomeB1_DRC_plus_L5M_P,hb1.HomeB1_Kpct_L5M_P,hb1.HomeB1_BBpct_L5M_P,
hb2.HomeB2_PA_L5M_P,hb2.HomeB2_OBP_L5M_P,hb2.HomeB2_OPS_L5M_P,hb2.HomeB2_DRC_plus_L5M_P,hb2.HomeB2_Kpct_L5M_P,hb2.HomeB2_BBpct_L5M_P,
hb3.HomeB3_PA_L5M_P,hb3.HomeB3_OBP_L5M_P,hb3.HomeB3_OPS_L5M_P,hb3.HomeB3_DRC_plus_L5M_P,hb3.HomeB3_Kpct_L5M_P,hb3.HomeB3_BBpct_L5M_P,
hb4.HomeB4_PA_L5M_P,hb4.HomeB4_OBP_L5M_P,hb4.HomeB4_OPS_L5M_P,hb4.HomeB4_DRC_plus_L5M_P,hb4.HomeB4_Kpct_L5M_P,hb4.HomeB4_BBpct_L5M_P,
hb5.HomeB5_PA_L5M_P,hb5.HomeB5_OBP_L5M_P,hb5.HomeB5_OPS_L5M_P,hb5.HomeB5_DRC_plus_L5M_P,hb5.HomeB5_Kpct_L5M_P,hb5.HomeB5_BBpct_L5M_P,
hb6.HomeB6_PA_L5M_P,hb6.HomeB6_OBP_L5M_P,hb6.HomeB6_OPS_L5M_P,hb6.HomeB6_DRC_plus_L5M_P,hb6.HomeB6_Kpct_L5M_P,hb6.HomeB6_BBpct_L5M_P,
hb7.HomeB7_PA_L5M_P,hb7.HomeB7_OBP_L5M_P,hb7.HomeB7_OPS_L5M_P,hb7.HomeB7_DRC_plus_L5M_P,hb7.HomeB7_Kpct_L5M_P,hb7.HomeB7_BBpct_L5M_P,
hb8.HomeB8_PA_L5M_P,hb8.HomeB8_OBP_L5M_P,hb8.HomeB8_OPS_L5M_P,hb8.HomeB8_DRC_plus_L5M_P,hb8.HomeB8_Kpct_L5M_P,hb8.HomeB8_BBpct_L5M_P,
hb9.HomeB9_PA_L5M_P,hb9.HomeB9_OBP_L5M_P,hb9.HomeB9_OPS_L5M_P,hb9.HomeB9_DRC_plus_L5M_P,hb9.HomeB9_Kpct_L5M_P,hb9.HomeB9_BBpct_L5M_P,

ab1.AwayB1_PA_L5M_P,ab1.AwayB1_OBP_L5M_P,ab1.AwayB1_OPS_L5M_P,ab1.AwayB1_DRC_plus_L5M_P,ab1.AwayB1_Kpct_L5M_P,ab1.AwayB1_BBpct_L5M_P,
ab2.AwayB2_PA_L5M_P,ab2.AwayB2_OBP_L5M_P,ab2.AwayB2_OPS_L5M_P,ab2.AwayB2_DRC_plus_L5M_P,ab2.AwayB2_Kpct_L5M_P,ab2.AwayB2_BBpct_L5M_P,
ab3.AwayB3_PA_L5M_P,ab3.AwayB3_OBP_L5M_P,ab3.AwayB3_OPS_L5M_P,ab3.AwayB3_DRC_plus_L5M_P,ab3.AwayB3_Kpct_L5M_P,ab3.AwayB3_BBpct_L5M_P,
ab4.AwayB4_PA_L5M_P,ab4.AwayB4_OBP_L5M_P,ab4.AwayB4_OPS_L5M_P,ab4.AwayB4_DRC_plus_L5M_P,ab4.AwayB4_Kpct_L5M_P,ab4.AwayB4_BBpct_L5M_P,
ab5.AwayB5_PA_L5M_P,ab5.AwayB5_OBP_L5M_P,ab5.AwayB5_OPS_L5M_P,ab5.AwayB5_DRC_plus_L5M_P,ab5.AwayB5_Kpct_L5M_P,ab5.AwayB5_BBpct_L5M_P,
ab6.AwayB6_PA_L5M_P,ab6.AwayB6_OBP_L5M_P,ab6.AwayB6_OPS_L5M_P,ab6.AwayB6_DRC_plus_L5M_P,ab6.AwayB6_Kpct_L5M_P,ab6.AwayB6_BBpct_L5M_P,
ab7.AwayB7_PA_L5M_P,ab7.AwayB7_OBP_L5M_P,ab7.AwayB7_OPS_L5M_P,ab7.AwayB7_DRC_plus_L5M_P,ab7.AwayB7_Kpct_L5M_P,ab7.AwayB7_BBpct_L5M_P,
ab8.AwayB8_PA_L5M_P,ab8.AwayB8_OBP_L5M_P,ab8.AwayB8_OPS_L5M_P,ab8.AwayB8_DRC_plus_L5M_P,ab8.AwayB8_Kpct_L5M_P,ab8.AwayB8_BBpct_L5M_P,
ab9.AwayB9_PA_L5M_P,ab9.AwayB9_OBP_L5M_P,ab9.AwayB9_OPS_L5M_P,ab9.AwayB9_DRC_plus_L5M_P,ab9.AwayB9_Kpct_L5M_P,ab9.AwayB9_BBpct_L5M_P

FROM GameBoxScores_bphitting GBS

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB1_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB1_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB1_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB1_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB1_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB1_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter1IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb1 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB2_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB2_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB2_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB2_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB2_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB2_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter2IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb2 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB3_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB3_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB3_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB3_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB3_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB3_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter3IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb3 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB4_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB4_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB4_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB4_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB4_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB4_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter4IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb4 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB5_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB5_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB5_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB5_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB5_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB5_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter5IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb5 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB6_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB6_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB6_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB6_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB6_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB6_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter6IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb6 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB7_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB7_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB7_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB7_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB7_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB7_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter7IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb7 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB8_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB8_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB8_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB8_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB8_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB8_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter8IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb8 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB9_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB9_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB9_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB9_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB9_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB9_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter9IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb9 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB1_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB1_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB1_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB1_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB1_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB1_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter1IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab1 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB2_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB2_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB2_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB2_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB2_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB2_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter2IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab2 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB3_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB3_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB3_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB3_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB3_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB3_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter3IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab3 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB4_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB4_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB4_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB4_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB4_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB4_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter4IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab4 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB5_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB5_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB5_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB5_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB5_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB5_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter5IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab5 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB6_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB6_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB6_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB6_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB6_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB6_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter6IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab6 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB7_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB7_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB7_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB7_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB7_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB7_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter7IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab7 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB8_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB8_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB8_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB8_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB8_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB8_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter8IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab8 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB9_PA_L5M_P, SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB9_OBP_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB9_OPS_L5M_P,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB9_DRC_plus_L5M_P, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB9_Kpct_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB9_BBpct_L5M_P
    FROM(
        SELECT
        PA,OBP_W,OPS_W,DRC_plus_W,Kpct_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter9IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab9 on TRUE

""").df()



In [78]:
GameBoxScores_phitting.to_csv("GBSHitting.csv", index=False)

In [79]:
# IGNORE

uhs = pd.read_csv("/Users/owendrummond/Documents/python_projects/Capstone/uhs.csv")
uhs

,BatterID,Year
0,22266,2024
1,22275,2024
2,19913,2024
3,24703,2024
4,26151,2024
...,...,...
5861,22411,2025
5862,20450,2025
5863,22168,2025
5864,19294,2025


In [11]:
# Creation of batter game log table occurs in "projectR.ipynb". Contains all needed hitter game logs over desired time frame.

BatterGameLogFact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/BatterGameLogFact.csv')
BatterGameLogFact26 = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/BatterGameLogFact2026.csv')
BatterGameLogFact26

,PlayerName,playerid,Date,Team,Opp,season,Age,BatOrder,Pos,G,...,KNv,wKN,wKN/C,pfxUN%,piCS%,pivCS,piCS-X,piCS-Z,piwCS,piwCS/C
0,Xavier Edwards,22266,2026-04-22,MIA,STL,2026,26,2,DH,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Xavier Edwards,22266,2026-04-21,MIA,STL,2026,26,4,2B,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Xavier Edwards,22266,2026-04-20,MIA,STL,2026,26,4,2B,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Xavier Edwards,22266,2026-04-19,MIA,MIL,2026,26,2,2B,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Xavier Edwards,22266,2026-04-18,MIA,MIL,2026,26,2,2B,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7800,Walbert Urena,29357,2026-03-28,LAA,@HOU,2026,22,0,P,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7801,Walbert Urena,29357,2026-03-26,LAA,@HOU,2026,22,0,P,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7802,Tatsuya Imai,37124,2026-04-10,HOU,@SEA,2026,28,0,P,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7803,Tatsuya Imai,37124,2026-04-04,HOU,@ATH,2026,28,0,P,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# STILL IN PROGRESS FOR LIVE ADAPTATION

common_cols1 = list(set(BatterGameLogFact.columns) & set(BatterGameLogFact26.columns))

Batter_Combined_Fact = pd.concat([
    BatterGameLogFact[common_cols1], 
    BatterGameLogFact26[common_cols1]
], ignore_index=True)

Batter_Combined_Fact

,pfxCH-X,SFv,piwSI,Year,1B,playerid,piZ-Swing%,piFS-Z,pfxwST/C,scW-Zone%,...,wCB/C,wSF/C,maxEV,scSO-Contact%,xwOBA,piwFC/C,Swing%,pfxaaSL,piCH-Z,piSwing%
0,9.594926,NaN,0.396797,2024,2,22266,0.714286,NaN,NaN,0.083333,...,-24.982324,NaN,89.4,1.00,0.325750,NaN,0.500000,54.033241,0.122419,0.500000
1,-8.636301,87.5,NaN,2024,0,22266,0.700000,0.980121,NaN,0.043478,...,33.476341,-8.757003,95.8,NaN,0.342426,3.324525,0.434783,41.404121,1.827350,0.434783
2,-8.362905,NaN,1.072792,2024,1,22266,0.777778,NaN,NaN,0.095238,...,-8.213652,NaN,100.9,1.00,0.167604,68.784463,0.523810,47.309090,2.371455,0.523810
3,-8.101519,96.0,0.094794,2024,1,22266,0.625000,NaN,NaN,0.176471,...,NaN,3.440772,93.3,0.75,0.363210,NaN,0.411765,51.880322,5.174970,0.411765
4,-9.622075,87.0,-0.016696,2024,2,22266,0.785714,0.398684,NaN,0.047619,...,43.166608,-1.297190,88.8,NaN,0.238000,-5.477396,0.619048,NaN,1.516632,0.619048
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
398565,NaN,NaN,NaN,2026,0,29357,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
398566,NaN,NaN,NaN,2026,0,29357,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
398567,NaN,NaN,NaN,2026,0,37124,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
398568,NaN,NaN,NaN,2026,0,37124,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
Batter_Combined_Fact.to_csv('Batter_Combined_Fact.csv', index=False)

In [112]:
# Adding last 10 appearances hitting data to GameBoxScores for each lineup spot.

GameBoxScoresDPH = duckdb.query("""
with stats_weighted as (
    SELECT
        playerid,season,AB,PA,Events,Date,(AVG * AB) as AVG_W,("BB%" * PA) as BB_pct_W,("K%" * PA) as K_pct_W,("BB/K" * PA) as BB_K_W,(OBP * PA) as OBP_W,(OPS * PA) as OPS_W,("GB/FB" * Events) as GB_FB_W,
        (wOBA * PA) as wOBA_W,("wRC+" * PA) as wRC_plus_W,(xwOBA * PA) as xwOBA_W,(xAVG * AB) as xAVG_W,(xSLG * AB) as xSLG_W,("Barrel%" * Events) as Barrel_pct_W
    FROM BatterGameLogFact
)

select GBS.*,
    HomeB1_PA_L10A,HomeB1_AVG_L10A,HomeB1_BB_pct_L10A,HomeB1_K_pct_L10A,HomeB1_BB_K_L10A,HomeB1_OBP_L10A,HomeB1_OPS_L10A,HomeB1_GB_FB_L10A,HomeB1_wOBA_L10A,HomeB1_wRC_plus_L10A,HomeB1_xwOBA_L10A,
    HomeB1_xAVG_L10A,HomeB1_xSLG_L10A,HomeB1_Barrel_pct_L10A,
    HomeB2_PA_L10A,HomeB2_AVG_L10A,HomeB2_BB_pct_L10A,HomeB2_K_pct_L10A,HomeB2_BB_K_L10A,HomeB2_OBP_L10A,HomeB2_OPS_L10A,HomeB2_GB_FB_L10A,HomeB2_wOBA_L10A,HomeB2_wRC_plus_L10A,HomeB2_xwOBA_L10A,
    HomeB2_xAVG_L10A,HomeB2_xSLG_L10A,HomeB2_Barrel_pct_L10A,
    HomeB3_PA_L10A,HomeB3_AVG_L10A,HomeB3_BB_pct_L10A,HomeB3_K_pct_L10A,HomeB3_BB_K_L10A,HomeB3_OBP_L10A,HomeB3_OPS_L10A,HomeB3_GB_FB_L10A,HomeB3_wOBA_L10A,HomeB3_wRC_plus_L10A,HomeB3_xwOBA_L10A,
    HomeB3_xAVG_L10A,HomeB3_xSLG_L10A,HomeB3_Barrel_pct_L10A,
    HomeB4_PA_L10A,HomeB4_AVG_L10A,HomeB4_BB_pct_L10A,HomeB4_K_pct_L10A,HomeB4_BB_K_L10A,HomeB4_OBP_L10A,HomeB4_OPS_L10A,HomeB4_GB_FB_L10A,HomeB4_wOBA_L10A,HomeB4_wRC_plus_L10A,HomeB4_xwOBA_L10A,
    HomeB4_xAVG_L10A,HomeB4_xSLG_L10A,HomeB4_Barrel_pct_L10A,
    HomeB5_PA_L10A,HomeB5_AVG_L10A,HomeB5_BB_pct_L10A,HomeB5_K_pct_L10A,HomeB5_BB_K_L10A,HomeB5_OBP_L10A,HomeB5_OPS_L10A,HomeB5_GB_FB_L10A,HomeB5_wOBA_L10A,HomeB5_wRC_plus_L10A,HomeB5_xwOBA_L10A,
    HomeB5_xAVG_L10A,HomeB5_xSLG_L10A,HomeB5_Barrel_pct_L10A,
    HomeB6_PA_L10A,HomeB6_AVG_L10A,HomeB6_BB_pct_L10A,HomeB6_K_pct_L10A,HomeB6_BB_K_L10A,HomeB6_OBP_L10A,HomeB6_OPS_L10A,HomeB6_GB_FB_L10A,HomeB6_wOBA_L10A,HomeB6_wRC_plus_L10A,HomeB6_xwOBA_L10A,
    HomeB6_xAVG_L10A,HomeB6_xSLG_L10A,HomeB6_Barrel_pct_L10A,
    HomeB7_PA_L10A,HomeB7_AVG_L10A,HomeB7_BB_pct_L10A,HomeB7_K_pct_L10A,HomeB7_BB_K_L10A,HomeB7_OBP_L10A,HomeB7_OPS_L10A,HomeB7_GB_FB_L10A,HomeB7_wOBA_L10A,HomeB7_wRC_plus_L10A,HomeB7_xwOBA_L10A,
    HomeB7_xAVG_L10A,HomeB7_xSLG_L10A,HomeB7_Barrel_pct_L10A,
    HomeB8_PA_L10A,HomeB8_AVG_L10A,HomeB8_BB_pct_L10A,HomeB8_K_pct_L10A,HomeB8_BB_K_L10A,HomeB8_OBP_L10A,HomeB8_OPS_L10A,HomeB8_GB_FB_L10A,HomeB8_wOBA_L10A,HomeB8_wRC_plus_L10A,HomeB8_xwOBA_L10A,
    HomeB8_xAVG_L10A,HomeB8_xSLG_L10A,HomeB8_Barrel_pct_L10A,
    HomeB9_PA_L10A,HomeB9_AVG_L10A,HomeB9_BB_pct_L10A,HomeB9_K_pct_L10A,HomeB9_BB_K_L10A,HomeB9_OBP_L10A,HomeB9_OPS_L10A,HomeB9_GB_FB_L10A,HomeB9_wOBA_L10A,HomeB9_wRC_plus_L10A,HomeB9_xwOBA_L10A,
    HomeB9_xAVG_L10A,HomeB9_xSLG_L10A,HomeB9_Barrel_pct_L10A,
    AwayB1_PA_L10A,AwayB1_AVG_L10A,AwayB1_BB_pct_L10A,AwayB1_K_pct_L10A,AwayB1_BB_K_L10A,AwayB1_OBP_L10A,AwayB1_OPS_L10A,AwayB1_GB_FB_L10A,AwayB1_wOBA_L10A,AwayB1_wRC_plus_L10A,AwayB1_xwOBA_L10A,
    AwayB1_xAVG_L10A,AwayB1_xSLG_L10A,AwayB1_Barrel_pct_L10A,
    AwayB2_PA_L10A,AwayB2_AVG_L10A,AwayB2_BB_pct_L10A,AwayB2_K_pct_L10A,AwayB2_BB_K_L10A,AwayB2_OBP_L10A,AwayB2_OPS_L10A,AwayB2_GB_FB_L10A,AwayB2_wOBA_L10A,AwayB2_wRC_plus_L10A,AwayB2_xwOBA_L10A,
    AwayB2_xAVG_L10A,AwayB2_xSLG_L10A,AwayB2_Barrel_pct_L10A,
    AwayB3_PA_L10A,AwayB3_AVG_L10A,AwayB3_BB_pct_L10A,AwayB3_K_pct_L10A,AwayB3_BB_K_L10A,AwayB3_OBP_L10A,AwayB3_OPS_L10A,AwayB3_GB_FB_L10A,AwayB3_wOBA_L10A,AwayB3_wRC_plus_L10A,AwayB3_xwOBA_L10A,
    AwayB3_xAVG_L10A,AwayB3_xSLG_L10A,AwayB3_Barrel_pct_L10A,
    AwayB4_PA_L10A,AwayB4_AVG_L10A,AwayB4_BB_pct_L10A,AwayB4_K_pct_L10A,AwayB4_BB_K_L10A,AwayB4_OBP_L10A,AwayB4_OPS_L10A,AwayB4_GB_FB_L10A,AwayB4_wOBA_L10A,AwayB4_wRC_plus_L10A,AwayB4_xwOBA_L10A,
    AwayB4_xAVG_L10A,AwayB4_xSLG_L10A,AwayB4_Barrel_pct_L10A,
    AwayB5_PA_L10A,AwayB5_AVG_L10A,AwayB5_BB_pct_L10A,AwayB5_K_pct_L10A,AwayB5_BB_K_L10A,AwayB5_OBP_L10A,AwayB5_OPS_L10A,AwayB5_GB_FB_L10A,AwayB5_wOBA_L10A,AwayB5_wRC_plus_L10A,AwayB5_xwOBA_L10A,
    AwayB5_xAVG_L10A,AwayB5_xSLG_L10A,AwayB5_Barrel_pct_L10A,
    AwayB6_PA_L10A,AwayB6_AVG_L10A,AwayB6_BB_pct_L10A,AwayB6_K_pct_L10A,AwayB6_BB_K_L10A,AwayB6_OBP_L10A,AwayB6_OPS_L10A,AwayB6_GB_FB_L10A,AwayB6_wOBA_L10A,AwayB6_wRC_plus_L10A,AwayB6_xwOBA_L10A,
    AwayB6_xAVG_L10A,AwayB6_xSLG_L10A,AwayB6_Barrel_pct_L10A,
    AwayB7_PA_L10A,AwayB7_AVG_L10A,AwayB7_BB_pct_L10A,AwayB7_K_pct_L10A,AwayB7_BB_K_L10A,AwayB7_OBP_L10A,AwayB7_OPS_L10A,AwayB7_GB_FB_L10A,AwayB7_wOBA_L10A,AwayB7_wRC_plus_L10A,AwayB7_xwOBA_L10A,
    AwayB7_xAVG_L10A,AwayB7_xSLG_L10A,AwayB7_Barrel_pct_L10A,
    AwayB8_PA_L10A,AwayB8_AVG_L10A,AwayB8_BB_pct_L10A,AwayB8_K_pct_L10A,AwayB8_BB_K_L10A,AwayB8_OBP_L10A,AwayB8_OPS_L10A,AwayB8_GB_FB_L10A,AwayB8_wOBA_L10A,AwayB8_wRC_plus_L10A,AwayB8_xwOBA_L10A,
    AwayB8_xAVG_L10A,AwayB8_xSLG_L10A,AwayB8_Barrel_pct_L10A,
    AwayB9_PA_L10A,AwayB9_AVG_L10A,AwayB9_BB_pct_L10A,AwayB9_K_pct_L10A,AwayB9_BB_K_L10A,AwayB9_OBP_L10A,AwayB9_OPS_L10A,AwayB9_GB_FB_L10A,AwayB9_wOBA_L10A,AwayB9_wRC_plus_L10A,AwayB9_xwOBA_L10A,
    AwayB9_xAVG_L10A,AwayB9_xSLG_L10A,AwayB9_Barrel_pct_L10A

FROM GameBoxScores_phitting GBS

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB1_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as HomeB1_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB1_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB1_K_pct_L10A,
    SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB1_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB1_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB1_OPS_L10A,
    SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB1_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB1_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB1_wRC_plus_L10A,
    SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB1_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB1_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB1_xSLG_L10A,
    SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB1_Barrel_pct_L10A
    FROM(
        SELECT
        AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
        from stats_weighted sw
        where sw.playerid = GBS.HomeBatter1IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC
        limit 10
    )
) hb1 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB2_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as HomeB2_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB2_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB2_K_pct_L10A,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB2_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB2_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB2_OPS_L10A,
   SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB2_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB2_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB2_wRC_plus_L10A,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB2_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB2_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB2_xSLG_L10A,
   SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB2_Barrel_pct_L10A
   FROM(
       SELECT
       AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
       from stats_weighted sw
       where sw.playerid = GBS.HomeBatter2IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
       order by sw.Date DESC
       limit 10
   )
) hb2 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB3_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as HomeB3_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB3_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB3_K_pct_L10A,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB3_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB3_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB3_OPS_L10A,
   SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB3_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB3_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB3_wRC_plus_L10A,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB3_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB3_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB3_xSLG_L10A,
   SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB3_Barrel_pct_L10A
   FROM(
       SELECT
       AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
       from stats_weighted sw
       where sw.playerid = GBS.HomeBatter3IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
       order by sw.Date DESC
       limit 10
   )
) hb3 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB4_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as HomeB4_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB4_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB4_K_pct_L10A,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB4_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB4_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB4_OPS_L10A,
   SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB4_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB4_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB4_wRC_plus_L10A,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB4_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB4_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB4_xSLG_L10A,
   SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB4_Barrel_pct_L10A
   FROM(
       SELECT
       AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
       from stats_weighted sw
       where sw.playerid = GBS.HomeBatter4IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
       order by sw.Date DESC
       limit 10
   )
) hb4 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB5_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as HomeB5_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB5_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB5_K_pct_L10A,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB5_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB5_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB5_OPS_L10A,
   SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB5_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB5_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB5_wRC_plus_L10A,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB5_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB5_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB5_xSLG_L10A,
   SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB5_Barrel_pct_L10A
   FROM(
       SELECT
       AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
       from stats_weighted sw
       where sw.playerid = GBS.HomeBatter5IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
       order by sw.Date DESC
       limit 10
   )
) hb5 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB6_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as HomeB6_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB6_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB6_K_pct_L10A,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB6_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB6_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB6_OPS_L10A,
   SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB6_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB6_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB6_wRC_plus_L10A,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB6_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB6_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB6_xSLG_L10A,
   SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB6_Barrel_pct_L10A
   FROM(
       SELECT
       AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
       from stats_weighted sw
       where sw.playerid = GBS.HomeBatter6IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
       order by sw.Date DESC
       limit 10
   )
) hb6 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB7_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as HomeB7_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB7_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB7_K_pct_L10A,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB7_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB7_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB7_OPS_L10A,
   SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB7_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB7_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB7_wRC_plus_L10A,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB7_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB7_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB7_xSLG_L10A,
   SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB7_Barrel_pct_L10A
   FROM(
       SELECT
       AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
       from stats_weighted sw
       where sw.playerid = GBS.HomeBatter7IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
       order by sw.Date DESC
       limit 10
   )
) hb7 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB8_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as HomeB8_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB8_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB8_K_pct_L10A,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB8_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB8_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB8_OPS_L10A,
   SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB8_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB8_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB8_wRC_plus_L10A,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB8_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB8_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB8_xSLG_L10A,
   SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB8_Barrel_pct_L10A
   FROM(
       SELECT
       AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
       from stats_weighted sw
       where sw.playerid = GBS.HomeBatter8IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
       order by sw.Date DESC
       limit 10
   )
) hb8 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as HomeB9_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as HomeB9_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as HomeB9_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as HomeB9_K_pct_L10A,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB9_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as HomeB9_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB9_OPS_L10A,
   SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as HomeB9_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as HomeB9_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB9_wRC_plus_L10A,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as HomeB9_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB9_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB9_xSLG_L10A,
   SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as HomeB9_Barrel_pct_L10A
   FROM(
       SELECT
       AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
       from stats_weighted sw
       where sw.playerid = GBS.HomeBatter9IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
       order by sw.Date DESC
       limit 10
   )
) hb9 on TRUE

LEFT JOIN LATERAL(
   select
   SUM(PA) as AwayB1_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as AwayB1_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB1_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB1_K_pct_L10A,
   SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB1_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB1_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB1_OPS_L10A,
   SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB1_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB1_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB1_wRC_plus_L10A,
   SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB1_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB1_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB1_xSLG_L10A,
   SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB1_Barrel_pct_L10A
   FROM(
       SELECT
       AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
       from stats_weighted sw
       where sw.playerid = GBS.AwayBatter1IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
       order by sw.Date DESC
       limit 10
   )
) ab1 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB2_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as AwayB2_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB2_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB2_K_pct_L10A,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB2_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB2_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB2_OPS_L10A,
  SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB2_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB2_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB2_wRC_plus_L10A,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB2_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB2_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB2_xSLG_L10A,
  SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB2_Barrel_pct_L10A
  FROM(
      SELECT
      AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
      from stats_weighted sw
      where sw.playerid = GBS.AwayBatter2IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
      order by sw.Date DESC
      limit 10
  )
) ab2 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB3_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as AwayB3_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB3_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB3_K_pct_L10A,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB3_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB3_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB3_OPS_L10A,
  SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB3_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB3_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB3_wRC_plus_L10A,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB3_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB3_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB3_xSLG_L10A,
  SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB3_Barrel_pct_L10A
  FROM(
      SELECT
      AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
      from stats_weighted sw
      where sw.playerid = GBS.AwayBatter3IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
      order by sw.Date DESC
      limit 10
  )
) ab3 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB4_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as AwayB4_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB4_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB4_K_pct_L10A,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB4_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB4_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB4_OPS_L10A,
  SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB4_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB4_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB4_wRC_plus_L10A,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB4_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB4_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB4_xSLG_L10A,
  SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB4_Barrel_pct_L10A
  FROM(
      SELECT
      AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
      from stats_weighted sw
      where sw.playerid = GBS.AwayBatter4IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
      order by sw.Date DESC
      limit 10
  )
) ab4 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB5_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as AwayB5_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB5_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB5_K_pct_L10A,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB5_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB5_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB5_OPS_L10A,
  SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB5_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB5_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB5_wRC_plus_L10A,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB5_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB5_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB5_xSLG_L10A,
  SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB5_Barrel_pct_L10A
  FROM(
      SELECT
      AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
      from stats_weighted sw
      where sw.playerid = GBS.AwayBatter5IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
      order by sw.Date DESC
      limit 10
  )
) ab5 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB6_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as AwayB6_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB6_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB6_K_pct_L10A,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB6_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB6_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB6_OPS_L10A,
  SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB6_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB6_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB6_wRC_plus_L10A,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB6_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB6_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB6_xSLG_L10A,
  SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB6_Barrel_pct_L10A
  FROM(
      SELECT
      AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
      from stats_weighted sw
      where sw.playerid = GBS.AwayBatter6IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
      order by sw.Date DESC
      limit 10
  )
) ab6 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB7_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as AwayB7_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB7_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB7_K_pct_L10A,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB7_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB7_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB7_OPS_L10A,
  SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB7_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB7_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB7_wRC_plus_L10A,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB7_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB7_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB7_xSLG_L10A,
  SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB7_Barrel_pct_L10A
  FROM(
      SELECT
      AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
      from stats_weighted sw
      where sw.playerid = GBS.AwayBatter7IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
      order by sw.Date DESC
      limit 10
  )
) ab7 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB8_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as AwayB8_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB8_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB8_K_pct_L10A,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB8_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB8_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB8_OPS_L10A,
  SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB8_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB8_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB8_wRC_plus_L10A,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB8_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB8_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB8_xSLG_L10A,
  SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB8_Barrel_pct_L10A
  FROM(
      SELECT
      AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
      from stats_weighted sw
      where sw.playerid = GBS.AwayBatter8IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
      order by sw.Date DESC
      limit 10
  )
) ab8 on TRUE


LEFT JOIN LATERAL(
  select
  SUM(PA) as AwayB9_PA_L10A,SUM(AVG_W) / NULLIF(SUM(AB), 0) as AwayB9_AVG_L10A,SUM(BB_pct_W) / NULLIF(SUM(PA), 0) as AwayB9_BB_pct_L10A,SUM(K_pct_W) / NULLIF(SUM(PA), 0) as AwayB9_K_pct_L10A,
  SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB9_BB_K_L10A,SUM(OBP_W) / NULLIF(SUM(PA), 0) as AwayB9_OBP_L10A,SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB9_OPS_L10A,
  SUM(GB_FB_W) / NULLIF(SUM(Events), 0) as AwayB9_GB_FB_L10A,SUM(wOBA_W) / NULLIF(SUM(PA), 0) as AwayB9_wOBA_L10A,SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB9_wRC_plus_L10A,
  SUM(xwOBA_W) / NULLIF(SUM(PA), 0) as AwayB9_xwOBA_L10A,SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB9_xAVG_L10A,SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB9_xSLG_L10A,
  SUM(Barrel_pct_W) / NULLIF(SUM(Events), 0) as AwayB9_Barrel_pct_L10A
  FROM(
      SELECT
      AB,PA,Events,AVG_W,BB_pct_W,K_pct_W,BB_K_W,OBP_W,OPS_W,GB_FB_W,wOBA_W,wRC_plus_W,xwOBA_W,xAVG_W,xSLG_W,Barrel_pct_W
      from stats_weighted sw
      where sw.playerid = GBS.AwayBatter9IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
      order by sw.Date DESC
      limit 10
  )
) ab9 on TRUE

""").df()

In [117]:
GameBoxScoresDPH.to_csv("GameBoxScoresDPH.csv")

# Team Game Context

-----------
----------
----------

In [125]:
# Adding 2017 Game Data to construct win and run scoring trends (specifically for the 2018 games.)

bs2017 = pd.read_csv("/Users/owendrummond/Documents/python_projects/Capstone/gl2017 copy.txt", header=None)
bs2017 = clean_data_box(bs2017)
bs2017['AwayTeam'] = bs2017['AwayTeam'].replace(gbs_abbrs)
bs2017['HomeTeam'] = bs2017['HomeTeam'].replace(gbs_abbrs)
bs2017['Year'] = bs2017['Date'].astype(str).str[:4].astype('float64')
bs2017['Date'] = pd.to_datetime(bs2017['Date'].astype(str), format= '%Y%m%d')
bs2017

,Date,DayOfWeek,AwayTeam,AwayLeague,AwayGameNum,HomeTeam,HomeLeague,HomeGameNum,AwayScore,HomeScore,...,HomeBatter8ID,HomeBatter8,HomeBatter8Pos,HomeBatter9ID,HomeBatter9,HomeBatter9Pos,Finished,HomeTeamWin,AwayTeamWin,Year
0,2017-04-02,Sun,SFG,NL,1,ARI,NL,1,5,6,...,mathj001,Jeff Mathis,2,greiz001,Zack Greinke,1,Y,1,0,2017.0
1,2017-04-02,Sun,CHC,NL,1,STL,NL,1,3,4,...,gricr001,Randal Grichuk,7,martc006,Carlos Martinez,1,Y,1,0,2017.0
2,2017-04-02,Sun,NYY,AL,1,TBR,AL,1,3,7,...,smitm007,Mallex Smith,7,norrd001,Derek Norris,2,Y,1,0,2017.0
3,2017-04-03,Mon,PHI,NL,1,CIN,NL,1,4,3,...,barnt001,Tucker Barnhart,2,felds001,Scott Feldman,1,Y,0,1,2017.0
4,2017-04-03,Mon,SDP,NL,1,LAD,NL,1,3,14,...,puigy001,Yasiel Puig,9,kersc001,Clayton Kershaw,1,Y,1,0,2017.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2425,2017-10-01,Sun,ARI,NL,162,KCR,AL,162,14,2,...,buted001,Drew Butera,2,cainl001,Lorenzo Cain,8,Y,0,1,2017.0
2426,2017-10-01,Sun,DET,AL,162,MIN,AL,162,1,5,...,castj006,Jason Castro,2,grosr001,Robbie Grossman,9,Y,1,0,2017.0
2427,2017-10-01,Sun,TOR,AL,162,NYY,AL,162,2,1,...,frazc001,Clint Frazier,7,austt001,Tyler Austin,9,Y,0,1,2017.0
2428,2017-10-01,Sun,BAL,AL,162,TBR,AL,162,0,6,...,smitm007,Mallex Smith,8,robed004,Daniel Robertson,6,Y,1,0,2017.0


In [132]:
duckdb.execute("CREATE TABLE GBS_Master AS SELECT * FROM GameBoxScoresDPH")

In [133]:
# 1. Get the shared columns as before
gbs_cols = duckdb.query("PRAGMA table_info('GBS_Master')").df()['name'].tolist()
bs_cols = bs2017.columns.tolist()
shared_cols = [c for c in bs_cols if c in gbs_cols]

quoted_cols = [f'"{c}"' for c in shared_cols]
col_string = ", ".join(quoted_cols)

# 3. Execute the insert
sql = f"INSERT INTO GBS_Master ({col_string}) SELECT {col_string} FROM bs2017"
duckdb.execute(sql)

In [134]:
# Pull the entire Master table back into a Pandas DataFrame
GBS_final = duckdb.query("SELECT * FROM GBS_Master").df()

In [135]:
GBS_final

,GAMEID,Unnamed: 0,Date,DayOfWeek,AwayTeam,AwayLeague,AwayGameNum,HomeTeam,HomeLeague,HomeGameNum,...,AwayB9_BB_K_L10A,AwayB9_OBP_L10A,AwayB9_OPS_L10A,AwayB9_GB_FB_L10A,AwayB9_wOBA_L10A,AwayB9_wRC_plus_L10A,AwayB9_xwOBA_L10A,AwayB9_xAVG_L10A,AwayB9_xSLG_L10A,AwayB9_Barrel_pct_L10A
0,9928,10946,2023-04-23,Sun,CHW,AL,22,TBR,AL,22,...,0.156250,0.187500,0.476562,0.826087,0.205980,30.103186,0.208916,0.164343,0.290429,0.043478
1,969,10986,2023-04-26,Wed,NYY,AL,25,MIN,AL,25,...,0.067568,0.162162,0.405405,1.603448,0.159337,3.668582,0.248987,0.227624,0.374829,0.068966
2,10149,11242,2023-05-16,Tue,TBR,AL,43,NYM,NL,43,...,0.441176,0.294118,0.725490,0.850000,0.314211,106.153943,0.332971,0.193115,0.566621,0.250000
3,10221,11331,2023-05-22,Mon,TOR,AL,48,TBR,AL,49,...,0.230769,0.461538,1.179487,2.177419,0.496795,225.911672,0.359408,0.300037,0.471482,0.064516
4,10259,11381,2023-05-26,Fri,WAS,NL,51,KCR,AL,52,...,0.081081,0.243243,0.716216,0.820513,0.299540,92.053906,0.355200,0.275488,0.550050,0.115385
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20331,<NA>,<NA>,2017-10-01 00:00:00,Sun,ARI,NL,162,KCR,AL,162,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20332,<NA>,<NA>,2017-10-01 00:00:00,Sun,DET,AL,162,MIN,AL,162,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20333,<NA>,<NA>,2017-10-01 00:00:00,Sun,TOR,AL,162,NYY,AL,162,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20334,<NA>,<NA>,2017-10-01 00:00:00,Sun,BAL,AL,162,TBR,AL,162,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [138]:
# Joining GBS on itself to derive win and run scoring trend data. At the end I filter out 2017 data as it has served its purpose.

GameBoxScoresFinal = duckdb.query("""
WITH TeamWise as(
    select
        HomeTeam as Team,
        Date,
        HomeScore as RunsScored,
        AwayScore as RunsAllowed,
        HomeTeamWin as IsWin,
        OutTotal
    FROM GBS_final
    UNION ALL
    select
        AwayTeam as Team,
        Date,
        AwayScore as RunsScored,
        HomeScore as RunsAllowed,
        AwayTeamWin as IsWin,
        OutTotal
    FROM GBS_final
),

main as (

select GBS.*,

AL162.AwayTeamWinsL162, 
HL162.HomeTeamWinsL162,
AL50.AwayTeamWinsL50, 
HL50.HomeTeamWinsL50,
AL10.AwayTeamWinsL10, AL10.AwayTeamRunsScoredL10, AL10.AwayTeamRunsAllowedL10,
HL10.HomeTeamWinsL10, HL10.HomeTeamRunsScoredL10, HL10.HomeTeamRunsAllowedL10,
AL100.AwayTeamRunsScoredL100, AL100.AwayTeamRunsAllowedL100,
HL100.HomeTeamRunsScoredL100, HL100.HomeTeamRunsAllowedL100,
AL3.AwayTeamAvgOutTotalL3, 
HL3.HomeTeamAvgOutTotalL3

FROM GBS_final GBS

LEFT JOIN LATERAL(
    select
        SUM(IsWin) as AwayTeamWinsL162
    from(
        select
            IsWin
        from TeamWise tw
        where tw.Team = GBS.AwayTeam and CAST(tw.Date as date) < CAST(GBS.Date as date)
        order by tw.Date DESC
        limit 162
    )
) AL162 on true

LEFT JOIN LATERAL(
    select
        SUM(IsWin) as HomeTeamWinsL162
    from(
        select
            IsWin
        from TeamWise tw
        where tw.Team = GBS.HomeTeam and CAST(tw.Date as date) < CAST(GBS.Date as date)
        order by tw.Date DESC
        limit 162
    )
) HL162 on true

LEFT JOIN LATERAL(
    select
        SUM(IsWin) as AwayTeamWinsL50
    from(
        select
            IsWin
        from TeamWise tw
        where tw.Team = GBS.AwayTeam and CAST(tw.Date as date) < CAST(GBS.Date as date)
        order by tw.Date DESC
        limit 50
    )
) AL50 on true

LEFT JOIN LATERAL(
    select
        SUM(IsWin) as HomeTeamWinsL50
    from(
        select
            IsWin
        from TeamWise tw
        where tw.Team = GBS.HomeTeam and CAST(tw.Date as date) < CAST(GBS.Date as date)
        order by tw.Date DESC
        limit 50
    )
) HL50 on true

LEFT JOIN LATERAL(
    select
        SUM(IsWin) as AwayTeamWinsL10, SUM(RunsScored) as AwayTeamRunsScoredL10, SUM(RunsAllowed) as AwayTeamRunsAllowedL10
    from(
        select
            IsWin, RunsScored, RunsAllowed
        from TeamWise tw
        where tw.Team = GBS.AwayTeam and CAST(tw.Date as date) < CAST(GBS.Date as date)
        order by tw.Date DESC
        limit 10
    )
) AL10 on true

LEFT JOIN LATERAL(
    select
        SUM(IsWin) as HomeTeamWinsL10, SUM(RunsScored) as HomeTeamRunsScoredL10, SUM(RunsAllowed) as HomeTeamRunsAllowedL10
    from(
        select
            IsWin, RunsScored, RunsAllowed
        from TeamWise tw
        where tw.Team = GBS.HomeTeam and CAST(tw.Date as date) < CAST(GBS.Date as date)
        order by tw.Date DESC
        limit 10
    )
) HL10 on true

LEFT JOIN LATERAL(
    select
        SUM(RunsScored) as AwayTeamRunsScoredL100, SUM(RunsAllowed) as AwayTeamRunsAllowedL100
    from(
        select
            RunsScored, RunsAllowed
        from TeamWise tw
        where tw.Team = GBS.AwayTeam and CAST(tw.Date as date) < CAST(GBS.Date as date)
        order by tw.Date DESC
        limit 100
    )
) AL100 on true

LEFT JOIN LATERAL(
    select
        SUM(RunsScored) as HomeTeamRunsScoredL100, SUM(RunsAllowed) as HomeTeamRunsAllowedL100
    from(
        select
           RunsScored, RunsAllowed
        from TeamWise tw
        where tw.Team = GBS.HomeTeam and CAST(tw.Date as date) < CAST(GBS.Date as date)
        order by tw.Date DESC
        limit 100
    )
) HL100 on true

LEFT JOIN LATERAL(
    select
        AVG(OutTotal) as AwayTeamAvgOutTotalL3
    from(
        select
            OutTotal
        from TeamWise tw
        where tw.Team = GBS.AwayTeam and CAST(tw.Date as date) < CAST(GBS.Date as date)
        order by tw.Date DESC
        limit 3
    )
) AL3 on true

LEFT JOIN LATERAL(
    select
        AVG(OutTotal) as HomeTeamAvgOutTotalL3
    from(
        select
            OutTotal
        from TeamWise tw
        where tw.Team = GBS.HomeTeam and CAST(tw.Date as date) < CAST(GBS.Date as date)
        order by tw.Date DESC
        limit 3
    )
) HL3 on true

)

select * from main
where Year != 2017
order by Date

""").df()

In [139]:
GameBoxScoresFinal.to_csv("GameBoxScoresFinal.csv", index= False)

In [142]:
print(GameBoxScoresFinal.columns.tolist())

['GAMEID', 'Unnamed: 0', 'Date', 'DayOfWeek', 'AwayTeam', 'AwayLeague', 'AwayGameNum', 'HomeTeam', 'HomeLeague', 'HomeGameNum', 'AwayScore', 'HomeScore', 'OutTotal', 'Day/Night', 'ParkID', 'Attendence', 'LengthMin', 'AwayLineScore', 'HomeLineScore', 'AwayPitchersUsed', 'HomePitchersUsed', 'AwayPitcherID', 'AwayPitcher', 'HomePitcherID', 'HomePitcher', 'AwayBatter1ID', 'AwayBatter1', 'AwayBatter1Pos', 'AwayBatter2ID', 'AwayBatter2', 'AwayBatter2Pos', 'AwayBatter3ID', 'AwayBatter3', 'AwayBatter3Pos', 'AwayBatter4ID', 'AwayBatter4', 'AwayBatter4Pos', 'AwayBatter5ID', 'AwayBatter5', 'AwayBatter5Pos', 'AwayBatter6ID', 'AwayBatter6', 'AwayBatter6Pos', 'AwayBatter7ID', 'AwayBatter7', 'AwayBatter7Pos', 'AwayBatter8ID', 'AwayBatter8', 'AwayBatter8Pos', 'AwayBatter9ID', 'AwayBatter9', 'AwayBatter9Pos', 'HomeBatter1ID', 'HomeBatter1', 'HomeBatter1Pos', 'HomeBatter2ID', 'HomeBatter2', 'HomeBatter2Pos', 'HomeBatter3ID', 'HomeBatter3', 'HomeBatter3Pos', 'HomeBatter4ID', 'HomeBatter4', 'HomeBatter4Po

In [159]:
# IGNORE. This Excel file was overwritten so I do not have access to it anymore. See rookieDF.csv in files.

rookieDF = pd.read_excel('/Users/owendrummond/Desktop/Excel/Rookies.xlsx', sheet_name='AllBatters')
rookieDF

,#,Season,Name,Team,G,PA,HR,R,RBI,SB,...,SLG,wOBA,xwOBA,wRC+,BsR,Off,Def,WAR,top100,top20
0,1,2018,Ronald Acuña Jr.,ATL,111,487,26,78,64,16,...,0.552,0.388,0.375,142.0,2.2,27.4,0.4,4.4,1.0,1.0
1,2,2018,Joey Wendle,TBR,139,545,7,62,61,16,...,0.435,0.338,0.304,117.0,3.3,14.6,5.4,4.0,NaN,NaN
2,3,2018,Miguel Andujar,NYY,149,606,27,83,92,2,...,0.527,0.361,0.322,129.0,-0.2,21.1,-4.2,3.9,1.0,NaN
3,4,2018,Harrison Bader,STL,138,427,12,61,37,15,...,0.422,0.326,0.286,107.0,4.5,8.1,14.8,3.7,NaN,NaN
4,5,2018,Juan Soto,WSN,116,494,22,77,70,5,...,0.517,0.392,0.372,146.0,0.9,28.4,-7.8,3.7,1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1390,194,2024,Dominic Fletcher,CHW,72,241,1,14,17,0,...,0.256,0.227,0.247,43.0,-0.5,-16.2,0.4,-0.8,NaN,NaN
1391,195,2024,Justin Foscue,TEX,15,44,0,3,1,0,...,0.071,0.080,0.132,-60.0,0.0,-8.1,-1.3,-0.8,NaN,NaN
1392,196,2024,Jordan Beck,COL,55,184,3,14,13,7,...,0.276,0.231,0.269,32.0,1.7,-13.3,-2.5,-1.0,NaN,NaN
1393,197,2024,Hunter Goodman,COL,70,224,13,24,36,1,...,0.417,0.274,0.282,61.0,-0.3,-10.6,-8.5,-1.2,NaN,NaN


In [160]:
url_list = ['https://www.fangraphs.com/prospects/the-board/2017-prospect-list', 'https://www.fangraphs.com/prospects/the-board/2018-prospect-list', 'https://www.fangraphs.com/prospects/the-board/2019-prospect-list', 
            'https://www.fangraphs.com/prospects/the-board/2020-prospect-list', 'https://www.fangraphs.com/prospects/the-board/2021-prospect-list', 'https://www.fangraphs.com/prospects/the-board/2022-prospect-list', 
            'https://www.fangraphs.com/prospects/the-board/2023-prospect-list', 'https://www.fangraphs.com/prospects/the-board/2024-prospect-list', 'https://www.fangraphs.com/prospects/the-board/2025-prospect-list', 
            'https://www.fangraphs.com/prospects/the-board/2026-prospect-list', ]

In [165]:
# Gather a list of top 50 prospects since 2017 using web scraping.

tp = []
for url in url_list:
    tp_temp = pd.read_html(url)
    
    tp_temp_df = tp_temp[6]
    
    tp.append(tp_temp_df)

tpdf17_26 = pd.concat(tp, ignore_index=True)

In [166]:
tpdf17_26

,Top 100Rank in all baseball,Org RkRank within team's farm system,NameName,OrgMLB Organization,"PosProjected defensive position or pitching role (Starter, Multi-, or Single-inning Relief)",Current LevelMost recent the level played at or had a transaction to,TrendFV change since last ranking,FVFuture Value (Explanation),ETAProjected year of debut/ loss of prospect eligibility,RiskDistance between ceiling and floor outcomes,...,BBats,TThrows,"SignedYear, round, organization",Signed FromSigned From,VideoVideo,ReportScouting report,Sign YrSign Yr,Sign MktSign Mkt,Sign OrgSign Org,BonusBonus
0,1,1.0,Yoán Moncada,CHW,INF,MLB,NaN,70,2017,NaN,...,S,R,j2 2015,Cuba,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1.0,Andrew Benintendi,BOS,OF,MLB,NaN,65,2017,NaN,...,L,L,1st rd 2015,Arkansas,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1.0,Amed Rosario,NYM,SS,MLB,NaN,65,2017,NaN,...,R,R,J2 2012,Dominican Republic,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1.0,Dansby Swanson,ATL,SS,MLB,NaN,65,2017,NaN,...,R,R,1st rd 2015,Vanderbilt,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1.0,Austin Meadows,PIT,OF,NaN,NaN,65,2017,NaN,...,L,L,1st rd 2013,Grayson HS (GA),NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,46,NaN,Eduardo Tait,MIN,C,A+,NaN,50,2029,High,...,L,R,NaN,Panama,NaN,NaN,2023.0,Intl15,PHI,$8k
496,47,1.0,Ethan Holliday,COL,3B,A,NaN,50,2029,High,...,L,R,NaN,Stillwater HS (OK),NaN,NaN,2025.0,Draft,COL,$9.0M
497,48,2.0,Arjun Nimmala,TOR,SS,A+,NaN,50,2028,High,...,R,R,NaN,Strawberry Crest HS (FL),NaN,NaN,2023.0,Draft,TOR,$3.0M
498,49,1.0,George Lombard Jr.,NYY,SS,AA,NaN,50,2027,High,...,R,R,NaN,Gulliver Schools (FL),NaN,NaN,2023.0,Draft,NYY,$3.3M


In [186]:
tpdf17_26.to_csv("topProspects.csv", index= False)

In [167]:
# Normalize: Strip whitespace and convert to lowercase
rookieDF['Name_Clean'] = rookieDF['Name'].str.strip().str.lower()
tpdf17_26['Name_Clean'] = tpdf17_26['NameName'].str.strip().str.lower()

# Re-run the lookup
top_50_names = set(tpdf17_26['Name_Clean'])
rookieDF['Top50_Indicator'] = rookieDF['Name_Clean'].isin(top_50_names).astype(int)

# Drop the helper column afterward
rookieDF.drop(columns=['Name_Clean'], inplace=True)

In [172]:
# View records where Top50 is 0 and Top20 is 1
rookieDF[(rookieDF['Top50_Indicator'] == 0) & (rookieDF['top20'] == 1)]

,#,Season,Name,Team,G,PA,HR,R,RBI,SB,...,wOBA,xwOBA,wRC+,BsR,Off,Def,WAR,top100,top20,Top50_Indicator


In [171]:
# Manually update the indicator for JJ Bleday
rookieDF.loc[rookieDF['Name'] == 'JJ Bleday', 'Top50_Indicator'] = 1

In [173]:
rookieDF.to_csv("rookieDF.csv", index=False)

In [175]:
# IGNORE. This Excel file was overwritten so I do not have access to it anymore. See RookieDFPitcher.csv in files.

rookieDFPitcher = pd.read_excel('/Users/owendrummond/Desktop/Excel/Rookies.xlsx', sheet_name='AllPitchers')
rookieDFPitcher

,index,Name,Team,W,L,SV,G,GS,IP,K_9,...,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45
0,1,Taylor Cole,LAA,0,0,0,2,2,2.1,15.43,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Chris Flexen,NYM,0,1,0,1,1,3.0,6.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Luke Farrell,CHC,0,2,0,2,2,6.0,9.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Jalen Beeks,BOS,0,1,0,1,1,4.0,9.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Evan Phillips,BAL,0,1,0,1,1,2.0,9.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
736,101,Luke Little,CHC,0,0,0,1,1,1.0,9.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
737,102,AJ Smith-Shawver,ATL,0,0,0,1,1,4.1,8.31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
738,103,Ben Joyce,LAA,0,0,0,1,1,2.0,13.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
739,104,Orion Kerkering,PHI,0,0,0,2,2,2.0,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [177]:
# Normalize: Strip whitespace and convert to lowercase
rookieDFPitcher['Name_Clean'] = rookieDFPitcher['Name'].str.strip().str.lower()
#tpdf17_26['Name_Clean'] = tpdf17_26['NameName'].str.strip().str.lower()

# Re-run the lookup
top_50_names = set(tpdf17_26['Name_Clean'])
rookieDFPitcher['Top50_Indicator'] = rookieDFPitcher['Name_Clean'].isin(top_50_names).astype(int)

# Drop the helper column afterward
rookieDFPitcher.drop(columns=['Name_Clean'], inplace=True)

In [180]:
rookieDFPitcher.to_csv("RookieDFPitcher.csv", index=False)

# NOTICE: EVERYTHING BELOW THIS POINT IS MISCELLANEOUS FIXES SUCH AS HANDLING DUPLICATE NAMES IN PLAYER_DIM, OR INSERTIG SOME NEW 2026 PLAYERS

In [185]:
player_dim.to_csv("player_dim.csv", index=False)

In [23]:
BPHitting26 = pd.read_csv('/Users/owendrummond/Downloads/BPHitting2026.csv')
BPPitching26 = pd.read_csv('/Users/owendrummond/Downloads/BPPitching2026.csv')
BPPitching26

,bpid,mlbid,Name,Age,Season,Team,G,GS,IP,W,...,DRA,FIP,cFIP,WHIP,K%,BB%,HR%,GB%,DRA-,WARP
0,158731,800311,Didier Fuentes,21,2026,ATL,2,1,7.0,0,...,3.96,2.74,91.0,1.57,33.3,6.1,3.0,40.0,87.0,0.1
1,150806,699134,Bradgley Rodriguez,22,2026,SD,10,0,12.3,0,...,3.52,1.95,90.0,1.05,24.5,6.1,0.0,58.8,78.0,0.2
2,150813,699151,Jedixson Paez,22,2026,CWS,3,0,3.0,0,...,6.11,14.79,123.0,2.33,0.0,18.8,12.5,30.8,135.0,0.0
3,151041,700712,Walbert Urena,22,2026,LAA,3,1,7.7,0,...,3.07,2.47,78.0,1.83,25.6,12.8,0.0,62.5,68.0,0.2
4,153121,702273,Noah Schultz,22,2026,CWS,2,2,9.3,1,...,4.24,3.98,97.0,0.96,26.3,13.2,2.6,30.4,94.0,0.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505,49727,472610,Luis García,39,2026,NYM,6,0,6.3,0,...,5.25,2.85,106.0,2.05,12.5,6.3,0.0,42.3,116.0,0.0
506,60922,573204,Caleb Thielbar,39,2026,CHC,10,0,8.7,2,...,5.19,3.51,103.0,1.15,30.6,11.1,2.8,23.8,115.0,0.0
507,54159,455119,Chris Martin,40,2026,TEX,8,0,6.3,1,...,5.00,3.44,105.0,1.74,24.1,0.0,3.4,19.0,110.0,0.0
508,56753,453286,Max Scherzer,41,2026,TOR,4,4,16.3,1,...,6.46,6.18,130.0,1.29,14.7,7.4,5.9,25.0,143.0,-0.2


In [24]:
rookies26 = pd.read_excel('/Users/owendrummond/Desktop/Excel/Rookies.xlsx', sheet_name='2026Rookies')
rookies26

ValueError: Worksheet named '2026Rookies' not found

In [30]:
# 1. Get a unique list of all valid IDs from your master dimension table
valid_ids = player_dim['mlb_id'].unique()

# 2. Filter BPPitching26: Keep only rows where 'mlbid' is NOT in the valid_ids list
BPPitching26 = BPPitching26[~BPPitching26['mlbid'].isin(valid_ids)]

# 3. Filter BPHitting26: Keep only rows where 'mlbid' is NOT in the valid_ids list
BPHitting26 = BPHitting26[~BPHitting26['mlbid'].isin(valid_ids)]

# --- Verification ---
print(f"Remaining Pitching MISSING: {len(BPPitching26)}")
print(f"Remaining Hitting MISSING:  {len(BPHitting26)}")

Remaining Pitching Orphans: 156
Remaining Hitting Orphans:  20


In [31]:
# Create a new version of the table containing only pitchers with at least 1 start
BPPitching26 = BPPitching26[BPPitching26['GS'] > 0]

# Verification
print(f"Pitching table now contains {len(BPPitching26)} pitchers with 1+ GS.")
print(f"Minimum GS in table: {BPPitching26['GS'].min()}")

Pitching table now contains 20 pitchers with 1+ GS.
Minimum GS in table: 1


In [32]:
pitchers_subset = BPPitching26[['mlbid', 'Name']].copy()
hitters_subset = BPHitting26[['mlbid', 'Name']].copy()

# 2. Stack them vertically
combined_players = pd.concat([pitchers_subset, hitters_subset], axis=0)

# 3. Remove duplicates to get a unique list of IDs
# This ensures that if a player appears in both tables, they only appear once here
combined_players = combined_players.drop_duplicates(subset=['mlbid'])

# 4. Final Clean-up: Reset the index for a fresh start
combined_players = combined_players.reset_index(drop=True)

print(f"Total unique players: {len(combined_players)}")
print(combined_players.head())

Total unique players to look up: 40
    mlbid            Name
0  700712   Walbert Urena
1  702273    Noah Schultz
2  691725  Andrew Painter
3  696270    Ryan Johnson
4  679883   Luinder Avila


In [38]:
rookies26ID = duckdb.query("""
        select cp.*,
        r.fgid, r.Team
        from combined_players cp
        left join rookies26 r on cp.Name = r.Name
""").df()
rookies26ID

,mlbid,Name,fgid,Team
0,805808,Kevin McGonigle,33572.0,DET
1,808959,Munetaka Murakami,37120.0,CHW
2,802139,JJ Wetherholt,34985.0,STL
3,800050,Chase DeLauter,32127.0,CLE
4,691740,Daniel Susac,31441.0,SFG
5,699912,Jose Fernandez,28020.0,ARI
6,702222,Justin Crawford,31791.0,PHI
7,803011,Sam Antonacci,35101.0,CHW
8,695020,Tanner Murray,28848.0,CHW
9,691620,Jeferson Quero,28272.0,MIL


In [40]:
rookies26ID = pd.read_excel('/Users/owendrummond/Desktop/Excel/Rookies.xlsx', sheet_name='NewRookieList')
rookies26ID

In [43]:
player_dim = duckdb.query("""
        select *
        from player_dim
        
        union 
        
        select Name as player_name, NULL as retro_id, mlbid as mlb_id, NULL as bref_id, fgid as fg_id
        from rookies26ID
""").df()
player_dim

,player_name,retro_id,mlb_id,bref_id,fg_id
0,Jose Fernandez,NaN,699912,NaN,28020
1,Justin Crawford,NaN,702222,NaN,31791
2,Foster Griffin,NaN,656492,NaN,16432
3,George Klassen,NaN,691946,NaN,33464
4,Nicky Lopez,lopen001,670032,lopezni01,19339
...,...,...,...,...,...
2517,Justin Hagenman,hagej002,663795,hagenju01,21546
2518,Victor Mederos,medev001,682989,medervi01,31533
2519,Gordon Graceffo,gracg001,700669,gracego01,29519
2520,Andrew Walters,walta001,689958,waltean01,33834


In [46]:
potIssues = duckdb.query("""
select player_name, count(*) as name_count
from player_dim group by player_name
order by name_count desc
""").df()
potIssues

,player_name,name_count
0,Luis Garcia,3
1,Jacob Wilson,2
2,Carlos Perez,2
3,Max Muncy,2
4,Luis Ortiz,2
...,...,...
2508,Bryan Baker,1
2509,Phil Hughes,1
2510,Austin Adams,1
2511,Osvaldo Bido,1


Jaocb Wilson (ATH): 33266
Max Muncy (LAD): 13301
Max Muncy (ATH): 29779
Luis Ortiz (CLE): 27646
Josh Smith (TEX): 26396
Logan Allen (CLE): 27589

In [44]:
player_dim.to_csv('player_dim.csv', index=False)

In [3]:
player_dim = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/player_dim.csv')
player_dim

,player_name,retro_id,mlb_id,bref_id,fg_id
0,Jose Fernandez,NaN,699912,NaN,28020
1,Justin Crawford,NaN,702222,NaN,31791
2,Foster Griffin,NaN,656492,NaN,16432
3,George Klassen,NaN,691946,NaN,33464
4,Nicky Lopez,lopen001,670032,lopezni01,19339
...,...,...,...,...,...
2517,Justin Hagenman,hagej002,663795,hagenju01,21546
2518,Victor Mederos,medev001,682989,medervi01,31533
2519,Gordon Graceffo,gracg001,700669,gracego01,29519
2520,Andrew Walters,walta001,689958,waltean01,33834


In [8]:
player_dim['Team'] = None
player_dim['Team'] = player_dim['Team'].astype(str)

team_mapping = {
    '33266': 'ATH', 
    '13301': 'LAD', 
    '29779': 'ATH', 
    '27646': 'CLE', 
    '26396': 'TEX', 
    '27589': 'CLE'
}

for fid, team_abbr in team_mapping.items():
    player_dim.loc[player_dim['fg_id'] == fid, 'Team'] = team_abbr

In [10]:
player_dim.to_csv('player_dim.csv', index=False)